In [ ]:
# ── CAPTURE CELL : Run this FIRST before any other cell ─────────────────────
# All print() output appears in BOTH the cell output AND ps1_outputs/console.log
# All plt.show() figures appear in BOTH the cell output AND ps1_outputs/<name>.png
import os, sys
OUT_DIR = "ps1_validator_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    """Writes to Jupyter cell output + log file simultaneously."""
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream   # Jupyter OutStream -> shows in cell
        self._log     = log_fh           # file handle       -> saved to disk
        self._is_tee  = True
    def write(self, data):
        try:
            self._jupyter.write(data)
            self._jupyter.flush()
        except Exception:
            pass
        try:
            self._log.write(data)
            self._log.flush()
        except Exception:
            pass
    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh    = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] All output -> cell output + " + os.path.join(OUT_DIR, "console.log"))
else:
    print("[capture] Already active.")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        # Counter stored as mutable default arg on the function itself --
        # immune to any variable-name collision from other cells in the notebook.
        def _show(*a, _cnt=[0], **k):
            for _fig_num in _plt.get_fignums():
                _fig = _plt.figure(_fig_num)
                _title = (
                    (_fig._suptitle.get_text().strip() if _fig._suptitle else None)
                    or (_fig.axes[0].get_title().strip() if _fig.axes else None)
                    or f"fig_{_cnt[0]:02d}"
                )
                _safe = "".join(
                    c if c.isalnum() or c in "-_ " else "_" for c in _title
                ).strip().replace(" ", "_")[:60]
                _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                try:
                    _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                    print(f"[capture] fig saved -> {_fname}")
                    _cnt[0] += 1
                except Exception as _save_err:
                    print(f"[capture] fig save failed: {_save_err}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
    print(f"[capture] matplotlib patched -> figures saved to {OUT_DIR}/")
except Exception as _cap_err:
    print("[capture] matplotlib not ready:", _cap_err)


## PS1 — VALIDATOR 3-day Failure Prediction

**Scope:** VALIDATOR (BMV) devices only (`mars_device_category = 'VALIDATOR'`)

**Label (R6-1 2026-07-17):**  
`will_fail_3d = 1` when a hardware OOS Set event exists in `silver.device_failures` (source: `device_event_enriched` where `is_oos_event=TRUE AND EVENT_STATE_TYPE_NAME='Set' AND mars_device_category='VALIDATOR'`) within the next 3 calendar days. Label is pre-computed in `device_ps1_daily__create.sql` — no notebook-level label derivation needed.

Label is now built in Gold SQL (`device_ps1_daily__create.sql`) via the `outage_label_days_3d` CTE using a `device_failures` UNION — the same CTEs as TVM/GATE but sourced from `device_failures` where `device_category = 'VALIDATOR'` instead of availability_events.

> **Changed from R5-1 2026-07-16** (deprecated): previously `will_fail_3d = 1` when the device's tap rejection rate exceeded its rolling 28-day mean + 2 SD in the next 3 days. That tap-anomaly label is replaced by the hardware OOS Set event label above.

**Pre-requisite:** run `run_layer_silver` (builds S26 `device_failures`) then `run_layer_gold` (builds `device_ps1_daily`) before running this notebook.  
Then verify: `SELECT round(100.0*avg(will_fail_3d),3) FROM mars_dev.gold.device_ps1_daily WHERE mars_device_category='VALIDATOR'` should be > 0.

**Weather:** Open-Meteo historical API (free, no key) — VALIDATOR bus devices are matched to facility lat/lon via FACILITY_ID from `dim_device`.

**Checkpoint dir:** `/home/sagemaker-user/PS1V/checkpoints`  
**MLflow experiment:** `chicago-ps1v-validator-failure`

**PS1 — Device Failure Prediction (3-day Binary) — VALIDATOR**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture : CUBIC MARS AWS v6
  Gold S3 (Parquet) → Weather enrichment (Open-Meteo) → Feature Store (offline+online)
  → Train 8 models + stacking → MLflow Experiment → MLflow Model Registry
  → SageMaker Endpoint → SageMaker Model Monitor (data quality + model quality)

Target  : will_fail_3d  (0=no failure within 3 days, 1=failure within 3 days)
Scope   : VALIDATOR only (hardware OOS Set event label, R6-1 2026-07-17)
Grain   : one row per DEVICE_ID per day
Run in  : SageMaker Studio JupyterLab  (kernel: Python 3 — Data Science 3.0)

---

In [ ]:
# ── CELL 1 : Install dependencies ───────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4",
    "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2",
    "optuna>=3.6", "shap>=0.45",
    "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0",
    "pandas>=2.0", "matplotlib", "seaborn",
    "imbalanced-learn",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("All packages installed.")

In [ ]:
# ── CELL 2 : Imports ────────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: tables exported as Parquet (DeltaProtocolError on deletionVectors)
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler, LabelEncoder
from sklearn.pipeline                import Pipeline
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import (RandomForestClassifier,
                                             StackingClassifier)
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, roc_curve,
                                             f1_score, precision_score, recall_score,
                                             average_precision_score, precision_recall_curve,
                                             confusion_matrix, classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn, mlflow.xgboost
from mlflow.models.signature         import infer_signature

try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
print("Imports OK.")

In [ ]:
# ── CELL 3 : Configuration ──────────────────────────────────────────────────────
# Fill in your artifact bucket name (the dedicated artefacts bucket)
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
GOLD_S3     = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION  = Session()
ROLE        = get_execution_role()
REGION      = boto3.Session().region_name
SM_CLIENT   = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT   = boto3.client("s3",        region_name=REGION)

# ── MLflow – SageMaker Managed Tracking Server ────────────────────────────
MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found – using local file fallback (EFS)")

mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

# Per-category names (experiment / model / registry / endpoint) are built
# inside CELL 6b and overridden at the top of each CATEGORY loop.
GOLD_VERSION = 6

# ── Feature Store ─────────────────────────────────────────────────────────
# Online store = latest record per device for real-time inference.
# IAM role now has AmazonSageMakerFeatureStoreAccess on the artifacts bucket.
ENABLE_FEATURE_STORE = True
FS_GROUP_NAME        = "ps1-chicago-device-features-dev"

# ── Target horizon ────────────────────────────────────────────────────────
# "3d" → predict failure within 3 days (tighter signal, higher precision)
# "7d" → predict failure within 7 days (more positives, easier AUC signal)
TARGET_HORIZON = "3d"

print(f"Region               : {REGION}")
print(f"Gold S3              : {GOLD_S3}")
print(f"Artifacts            : s3://{ARTIFACT_BUCKET}/")
print(f"MLflow URI           : {MLFLOW_TRACKING_URI}")
print(f"Enable Feature Store : {ENABLE_FEATURE_STORE}  ({FS_GROUP_NAME})")
print(f"Target horizon       : will_fail_{TARGET_HORIZON}")



# ── Evaluation Fix: operational constants (ported from PS1_Evaluation_Fix.ipynb) ──
DISPATCH_BUDGET_PCT = 0.25   # flag top 25% of fleet/day; adjust to maintenance capacity
SMOTE_RATIO         = 0.05   # VALIDATOR SMOTE target positive rate

# Recall floor per category (minimum recall the deployed model must achieve)
QUALITY_RECALL_FLOOR = {"VALIDATOR": 0.80}

# Planned maintenance event types for VALIDATOR (BMV) devices.
# These are excluded from will_fail_3d label in CELL 10.
# Source: device_event_enriched event_type_id; same shared planned codes as TVM/GATE
# plus VALIDATOR firmware/SW update codes. Refine as new event types are identified.
VALIDATOR_PLANNED_EVENT_TYPES = [105, 106, 108, 129, 135, 140, 507, 516, 522]

# Champion AP tolerance: use best ensemble only if it beats single LGB by > this
CHAMPION_AP_TOL = 0.002

print(f"DISPATCH_BUDGET_PCT  : {DISPATCH_BUDGET_PCT:.0%}")
print(f"SMOTE_RATIO          : {SMOTE_RATIO}")
print(f"QUALITY_RECALL_FLOOR : {QUALITY_RECALL_FLOOR}")
# -- Shared S3 aliases (used by CELL 6 weather + cross-wired output cell) --
BUCKET        = GOLD_BUCKET
SILVER_PREFIX = "chicago/silver"
GOLD_BASE     = "chicago/gold"
S3_OPTS       = {"client_kwargs": {"region_name": "us-east-1"}}


In [ ]:
# ── CELL 4 : Checkpoint — load saved state (skip Cells 5-10 if found) ───────────
# Saves: df, FEATURE_COLS, TARGET, CATEGORIES (arrays + metadata per TVM/GATE)
# Path : /home/sagemaker-user/PS1V/checkpoints/  (EFS — survives instance stop)
# Usage: Run Cells 1-3-3b → if CHECKPOINT_LOADED=True → jump to Cell 9 (Helpers)
#        Set FORCE_RELOAD=True to re-pull from S3 and rebuild features.

import os, joblib, pandas as _pd_ckpt

CKPT_DIR      = "/home/sagemaker-user/PS1V/checkpoints"
FORCE_RELOAD  = True
CHECKPOINT_LOADED = False

_ckpt_df   = os.path.join(CKPT_DIR, "df.parquet")
_ckpt_meta = os.path.join(CKPT_DIR, "meta.joblib")

if not FORCE_RELOAD and os.path.exists(_ckpt_df) and os.path.exists(_ckpt_meta):
    print("[CHECKPOINT] Loading from", CKPT_DIR)
    df      = _pd_ckpt.read_parquet(_ckpt_df)
    _meta   = joblib.load(_ckpt_meta)
    FEATURE_COLS = _meta["FEATURE_COLS"]
    TARGET       = _meta["TARGET"]
    ID_COLS      = _meta["ID_COLS"]
    _cat_meta    = _meta["CATEGORIES_meta"]
    _cat_arrays  = joblib.load(os.path.join(CKPT_DIR, "categories_arrays.joblib"))

    # Rebuild CATEGORIES dict — DataFrames from df filter + arrays from file
    CATEGORIES = {}
    for cat in ["VALIDATOR"]:
        if cat not in _cat_meta: continue
        cat_df = df[df["mars_device_category"] == cat].sort_values(
                    ["DEVICE_ID", "event_date"])
        tr = cat_df[cat_df["event_date"] <  "2025-10-01"]
        va = cat_df[(cat_df["event_date"] >= "2025-10-01") &
                    (cat_df["event_date"] <  "2026-01-01")]
        te = cat_df[cat_df["event_date"] >= "2026-01-01"]
        arrs = _cat_arrays[cat]
        CATEGORIES[cat] = {
            **_cat_meta[cat],
            "train_df": tr, "val_df": va, "test_df": te,
            "X_train": arrs["X_train"], "y_train": arrs["y_train"],
            "X_val":   arrs["X_val"],   "y_val":   arrs["y_val"],
            "X_test":  arrs["X_test"],  "y_test":  arrs["y_test"],
            "X_cv":    arrs["X_cv"],    "y_cv":    arrs["y_cv"],
        }
        print(f"  {cat}: train={len(tr):,}  val={len(va):,}  test={len(te):,}"
              f"  X_train={arrs['X_train'].shape}"
              f"  pos={CATEGORIES[cat]['pos']:,}")
        mlflow.set_experiment(CATEGORIES[cat]["experiment_name"])

    CHECKPOINT_LOADED = True
    print(f"\n  df={df.shape}  FEATURE_COLS={len(FEATURE_COLS)}  TARGET={TARGET}")
    print()
    print("  CHECKPOINT LOADED — skip Cells 4, 5, 6, 6b.")
    print("  Jump to Cell 7 (Feature Store) or Cell 9 (Helpers) if FS not needed.")
else:
    print("[CHECKPOINT] Not found or FORCE_RELOAD=True — running full pipeline (Cells 4-6b).")
    print("  Checkpoint will be saved automatically at end of Cell 6b.")


In [ ]:
# ── CELL 5 : Load Gold table (Parquet on S3) ────────────────────────────────────
# G01 source: mars_dev.gold.device_ps1_daily exported via notebooks/export_gold_to_s3.py
# Column note: gold SQL writes "transit_day"; aliased to "event_date" below.

print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# gold SQL writes transit_day; alias to event_date used throughout notebook
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['event_date'].min().date()}  →  {df['event_date'].max().date()}")
print(f"\nTarget distribution:\n{df['will_fail_3d'].value_counts()}")
print(f"Failure rate    : {df['will_fail_3d'].mean():.2%}")
print(f"\nDevice mix:\n{df.drop_duplicates('DEVICE_ID')['mars_device_category'].value_counts()}")
# Derive will_fail_7d ? not in Parquet; computed as forward 7-day window
# of chargeable_outage_count > 0. Used when TARGET_HORIZON = '7d'.
if 'will_fail_7d' not in df.columns and 'chargeable_outage_count' in df.columns:
    df = df.sort_values(['DEVICE_ID', 'event_date']).reset_index(drop=True)
    def _fwd_max(x, w=7):
        rev = x.values[::-1]
        fwd = __import__('pandas').Series(rev).rolling(w, min_periods=1).max().shift(1).values[::-1]
        return __import__('pandas').Series(fwd, index=x.index)
    df['will_fail_7d'] = (
        df.groupby('DEVICE_ID')['chargeable_outage_count'].transform(_fwd_max) > 0
    ).astype(int).fillna(0).astype(int)
    print('Derived will_fail_7d:')
    for _cat in df['mars_device_category'].dropna().unique():
        _r = df[df['mars_device_category']==_cat]['will_fail_7d'].mean()
        print(f'  {_cat}: {_r:.2%} positive rate')
elif 'will_fail_7d' in df.columns:
    print(f"will_fail_7d already in Parquet: {df['will_fail_7d'].mean():.2%} positive rate")
else:
    print('WARNING: chargeable_outage_count not found ? will_fail_7d cannot be derived.')


In [ ]:
# -- CELL 6 : Weather data enrichment -- Open-Meteo historical API ---------
# Fetches point-in-time daily weather per facility (no rolling — device-level
# rolling failure history is already in the gold table and handled in CELL 16).
# Facilities: all unique FACILITY_ID values in df (VALIDATOR/BMV devices).
# Lat/lon from dim_facility; falls back to Chicago centre for unmatched IDs.
# Columns added: wx_temperature_2m_max, wx_precipitation_sum, wx_wind_speed_10m_max

import requests, time

# Step 1: load facility -> lat/lon from dim_facility silver table
FACILITY_S3 = f"s3://{BUCKET}/{SILVER_PREFIX}/dim_facility"
try:
    fac_meta = pd.read_parquet(FACILITY_S3, storage_options=S3_OPTS,
                               columns=["FACILITY_ID","FACILITY_NAME","CITY","LATITUDE","LONGITUDE"])
    fac_meta = fac_meta.dropna(subset=["LATITUDE","LONGITUDE"]).drop_duplicates("FACILITY_ID")
except Exception as _e:
    print(f"dim_facility load failed ({_e}); using fallback lat/lon for all facilities.")
    fac_meta = pd.DataFrame(columns=["FACILITY_ID","FACILITY_NAME","CITY","LATITUDE","LONGITUDE"])

date_col    = "event_date" if "event_date" in df.columns else "transit_day"
active_fids = sorted(df["FACILITY_ID"].dropna().unique().tolist()) if "FACILITY_ID" in df.columns else []
fac_lookup  = fac_meta[fac_meta["FACILITY_ID"].isin(active_fids)].set_index("FACILITY_ID")

# Facility / city coverage report
print(f"FACILITY_ID values in gold table : {len(active_fids)}")
print(f"Matched with lat/lon in dim_fac  : {len(fac_lookup)}")
unmatched = [f for f in active_fids if f not in fac_lookup.index]
if unmatched:
    print(f"WARNING: {len(unmatched)} unmatched (will use Chicago fallback): {unmatched}")
if not fac_lookup.empty:
    disp = [c for c in ["FACILITY_NAME","CITY","LATITUDE","LONGITUDE"] if c in fac_lookup.columns]
    print("\nFacility list:")
    print(fac_lookup[disp].sort_values(disp[0] if disp else fac_lookup.columns[0]).to_string())
print()

CHICAGO_LAT, CHICAGO_LON = 41.8781, -87.6298
start_date = df[date_col].min().strftime("%Y-%m-%d")
end_date   = df[date_col].max().strftime("%Y-%m-%d")
print(f"Weather fetch range: {start_date} to {end_date}")

WX_VARS = "temperature_2m_max,precipitation_sum,wind_speed_10m_max"

# Step 2: fetch per-facility
facility_wx_frames, failed_fids = [], []
for fid in active_fids:
    if fid in fac_lookup.index:
        lat  = float(fac_lookup.at[fid, "LATITUDE"])
        lon  = float(fac_lookup.at[fid, "LONGITUDE"])
        city = str(fac_lookup.at[fid, "CITY"]) if "CITY" in fac_lookup.columns else "-"
    else:
        lat, lon, city = CHICAGO_LAT, CHICAGO_LON, "Chicago (fallback)"

    url = (f"https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={lat}&longitude={lon}"
           f"&start_date={start_date}&end_date={end_date}"
           f"&daily={WX_VARS}&timezone=America/Chicago")
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json().get("daily", {})
        wx = pd.DataFrame({
            "FACILITY_ID":           fid,
            date_col:                pd.to_datetime(data["time"]),
            "wx_temperature_2m_max": data.get("temperature_2m_max", []),
            "wx_precipitation_sum":  data.get("precipitation_sum",  []),
            "wx_wind_speed_10m_max": data.get("wind_speed_10m_max", []),
        })
        facility_wx_frames.append(wx)
        time.sleep(0.15)
    except Exception as wx_err:
        failed_fids.append(fid)
        print(f"  WARNING: weather failed for FACILITY_ID={fid} ({city}): {wx_err}")

if facility_wx_frames:
    weather_df = pd.concat(facility_wx_frames, ignore_index=True)
    df = df.merge(weather_df, on=["FACILITY_ID", date_col], how="left")
    WX_FEATURE_COLS = ["wx_temperature_2m_max", "wx_precipitation_sum", "wx_wind_speed_10m_max"]
    print(f"\nWeather joined: {len(active_fids) - len(failed_fids)}/{len(active_fids)} facilities OK")
    null_wx = df[WX_FEATURE_COLS].isnull().sum()
    if null_wx.any():
        print(f"NULL wx rows (devices missing facility match):")
        print(null_wx.to_string())
    print(df[WX_FEATURE_COLS].describe().round(2).to_string())
else:
    WX_FEATURE_COLS = []
    print("WARNING: no weather data fetched.")


In [ ]:
# ── CELL 7 : EDA ────────────────────────────────────────────────────────────────
# Fetches daily weather for each unique facility and merges into df by
# (facility_col, event_date).  Open-Meteo archive API is free, no key needed.
#
# Weather parameters added (all daily aggregates):
#   wx_temperature_2m_max/min/mean   -- thermal stress on electronics
#   wx_apparent_temperature_max/min  -- felt-temperature extremes
#   wx_precipitation_sum             -- total rain+snow (mm)
#   wx_rain_sum / wx_snowfall_sum    -- rain (mm) vs snow (cm)
#   wx_windspeed_10m_max             -- wind gusts (km/h)
#   wx_winddirection_10m_dominant    -- dominant wind direction (deg)
#   wx_relative_humidity_2m_max/min  -- corrosion / condensation risk
#   wx_dewpoint_2m_min               -- condensation risk
#   wx_surface_pressure_mean         -- barometric pressure (hPa)
#   wx_cloudcover_mean               -- cloud cover (%)
#   wx_shortwave_radiation_sum       -- solar radiation / UV (MJ/m2)
#   wx_et0_fao_evapotranspiration    -- evapotranspiration proxy
#
# Derived features (engineered from raw weather):
#   wx_temp_range_c      -- diurnal swing (thermal cycling stress)
#   wx_freeze_flag       -- temp_min < 0 C (freeze-thaw risk)
#   wx_heat_stress_flag  -- temp_max > 35 C
#   wx_extreme_precip_flag -- precipitation > 25 mm/day
#   wx_extreme_wind_flag   -- wind > 50 km/h
#   wx_high_humidity_flag  -- humidity > 90%
#   wx_precip_roll3d / wx_precip_roll7d -- rolling precipitation
#   wx_temp_mean_roll7d    -- 7-day rolling mean temperature
#   wx_freeze_thaw_roll7d  -- freeze-thaw cycles in past 7 days

import requests, time
import pandas as pd
import numpy as np

WEATHER_ADDED = False

if not CHECKPOINT_LOADED:

    # -- 1. Detect facility column ------------------------------------------
    _facility_col = next(
        (c for c in df.columns
         if any(k in c.lower() for k in ["facility", "station", "location", "site"])),
        None
    )

    _CITY_WEATHER_ONLY = False
    if _facility_col is None:
        # No facility/location column in Gold table -- try dim_device (silver S3).
        # dim_device maps DEVICE_ID -> FACILITY_NAME (~25 distinct CTA stations).
        _DIM_DEVICE_S3 = f"s3://{GOLD_BUCKET}/chicago/silver/dim_device"
        print(f"[WEATHER] No facility col in Gold -- loading dim_device: {_DIM_DEVICE_S3}")
        try:
            _dim_dev = pd.read_parquet(
                _DIM_DEVICE_S3,
                storage_options={"client_kwargs": {"region_name": REGION}},
                columns=["DEVICE_ID", "FACILITY_NAME", "FACILITY_ID"],
            )
            if "is_current" in _dim_dev.columns:
                _dim_dev = _dim_dev[_dim_dev["is_current"] == True]
            _dim_dev = _dim_dev.drop_duplicates("DEVICE_ID")[["DEVICE_ID", "FACILITY_NAME"]]
            df = df.merge(_dim_dev, on="DEVICE_ID", how="left")
            _facility_col = "FACILITY_NAME"
            _n_fac    = df[_facility_col].nunique()
            _pct_null = df[_facility_col].isna().mean()
            print(f"[WEATHER] FACILITY_NAME joined: {_n_fac} stations, {_pct_null:.1%} null")
        except Exception as _e:
            print(f"[WEATHER] dim_device load failed ({_e}) -- falling back to city-wide.")
            _CITY_COL = "__city__"
            df[_CITY_COL] = "chicago"
            _facility_col = _CITY_COL
            _CITY_WEATHER_ONLY = True

    if _facility_col:  # always True (detected / dim_device / city fallback)
        print(f"[WEATHER] Facility column: '{_facility_col}'")
        # VALIDATOR devices travel across facilities (bus routes); FACILITY_ID
        # reflects the primary facility assignment from dim_device — weather is
        # joined on that primary facility. Extract facilities from VALIDATOR rows only.
        _val_mask   = df["mars_device_category"] == "VALIDATOR"
        _facilities = df.loc[_val_mask, _facility_col].dropna().unique()
        print(f"[WEATHER] VALIDATOR unique facilities: {len(_facilities)}")

        # -- 2. Facility -> (lat, lon) lookup -----------------------------------
        # Chicago-area CTA / Metra / transit facility coordinates.
        # Add / override entries here for your specific facility names.
        FACILITY_COORDS = {
            # CTA Blue Line
            "o'hare":                  (41.9786, -87.9060),
            "ohare":                   (41.9786, -87.9060),
            "rosemont":                (41.9836, -87.8594),
            "jefferson park":          (41.9706, -87.7611),
            "logan square":            (41.9290, -87.7086),
            "clark/lake":              (41.8857, -87.6309),
            "washington/wells":        (41.8829, -87.6338),
            "quincy":                  (41.8786, -87.6338),
            "uic-halsted":             (41.8750, -87.6493),
            # CTA Red Line
            "howard":                  (42.0186, -87.6726),
            "wilson":                  (41.9647, -87.6577),
            "belmont (red)":           (41.9397, -87.6526),
            "fullerton":               (41.9252, -87.6527),
            "north/clybourn":          (41.9107, -87.6494),
            "chicago (red)":           (41.8964, -87.6285),
            "grand (red)":             (41.8916, -87.6275),
            "lake (red)":              (41.8847, -87.6277),
            "jackson (red)":           (41.8784, -87.6278),
            "roosevelt":               (41.8674, -87.6274),
            "cermak-chinatown":        (41.8530, -87.6305),
            "sox-35th":                (41.8312, -87.6298),
            "95th/dan ryan":           (41.7225, -87.6243),
            # CTA Brown/Orange/Green/Pink/Purple lines
            "kimball":                 (41.9677, -87.7134),
            "kedzie (brown)":          (41.9657, -87.7059),
            "francisco":               (41.9665, -87.6989),
            "rockwell":                (41.9657, -87.6905),
            "western (brown)":         (41.9656, -87.6827),
            "damen (brown)":           (41.9657, -87.6778),
            "montrose (brown)":        (41.9613, -87.6744),
            "irving park (brown)":     (41.9542, -87.6743),
            "addison (brown)":         (41.9475, -87.6743),
            "paulina":                 (41.9435, -87.6704),
            "southport":               (41.9435, -87.6636),
            "belmont (brown)":         (41.9398, -87.6528),
            "diversey":                (41.9323, -87.6531),
            "wellington":              (41.9267, -87.6531),
            "armitage":                (41.9184, -87.6531),
            "sedgwick":                (41.9107, -87.6387),
            "chicago (brown)":         (41.8965, -87.6362),
            "merchandise mart":        (41.8886, -87.6358),
            "midway":                  (41.7868, -87.7372),
            "harlem (orange)":         (41.7766, -87.6952),
            "ashland (orange)":        (41.7776, -87.6647),
            "35th/archer":             (41.8289, -87.6803),
            # Metra / suburbs
            "union station":           (41.8788, -87.6400),
            "ogilvie":                 (41.8827, -87.6414),
            "millennium":              (41.8847, -87.6244),
            "evanston":                (42.0451, -87.6877),
            "aurora":                  (41.7571, -88.3201),
            "joliet":                  (41.5250, -88.0817),
            "elgin":                   (42.0354, -88.2826),
            "waukegan":                (42.3636, -87.8448),
            # Generic fallback
            "downtown chicago":        (41.8781, -87.6298),
            "chicago":                 (41.8781, -87.6298),
            "loop":                    (41.8827, -87.6323),
            "default":                 (41.8781, -87.6298),
        }

        def _get_coords(name):
            if pd.isna(name): return FACILITY_COORDS["default"]
            nl = str(name).lower().strip()
            if nl in FACILITY_COORDS: return FACILITY_COORDS[nl]
            for k, v in FACILITY_COORDS.items():
                if k != "default" and (k in nl or nl in k): return v
            return FACILITY_COORDS["default"]

        # -- 3. Fetch Open-Meteo archive weather --------------------------------
        _date_min  = df["event_date"].min().strftime("%Y-%m-%d")
        _date_max  = df["event_date"].max().strftime("%Y-%m-%d")
        _WX_PARAMS = [
            "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
            "apparent_temperature_max", "apparent_temperature_min",
            "precipitation_sum", "rain_sum", "snowfall_sum",
            "windspeed_10m_max", "winddirection_10m_dominant",
            "relative_humidity_2m_max", "relative_humidity_2m_min",
            "dewpoint_2m_min", "surface_pressure_mean",
            "cloudcover_mean", "shortwave_radiation_sum",
            "et0_fao_evapotranspiration",
        ]
        _WX_URL   = "https://archive-api.open-meteo.com/v1/archive"
        _wx_frames = []
        _seen_coords = {}   # cache: deduplicate API calls for same location

        for _fac in _facilities:
            _lat, _lon = _get_coords(_fac)
            _coord_key = (round(_lat, 3), round(_lon, 3))

            if _coord_key in _seen_coords:
                _wx_day = _seen_coords[_coord_key].copy()
            else:
                try:
                    _resp = requests.get(_WX_URL, params={
                        "latitude":   _lat,
                        "longitude":  _lon,
                        "start_date": _date_min,
                        "end_date":   _date_max,
                        "daily":      ",".join(_WX_PARAMS),
                        "timezone":   "America/Chicago",
                    }, timeout=30)
                    _resp.raise_for_status()
                    _json   = _resp.json()
                    _wx_day = pd.DataFrame(_json["daily"])
                    _wx_day["time"] = pd.to_datetime(_wx_day["time"])
                    _seen_coords[_coord_key] = _wx_day.copy()
                    time.sleep(0.15)   # polite rate-limit
                except Exception as _e:
                    print(f"  [WEATHER] WARNING: API call failed for '{_fac}' ({_lat},{_lon}): {_e}")
                    continue

            _wx_day = _wx_day.copy()
            _wx_day[_facility_col] = _fac
            _wx_frames.append(_wx_day)

        if not _wx_frames:
            print("[WEATHER] No weather data fetched -- skipping merge.")
        else:
            _wx_all = pd.concat(_wx_frames, ignore_index=True)
            _wx_all = _wx_all.rename(columns={"time": "event_date"})

            # Prefix raw weather cols with wx_
            _raw_wx = [c for c in _wx_all.columns if c not in ["event_date", _facility_col]]
            _wx_all = _wx_all.rename(columns={c: f"wx_{c}" for c in _raw_wx})

            # -- 4. Derived weather features ---------------------------------
            _wx_all["wx_temp_range_c"]         = (_wx_all["wx_temperature_2m_max"]
                                                   - _wx_all["wx_temperature_2m_min"])
            _wx_all["wx_freeze_flag"]          = (_wx_all["wx_temperature_2m_min"] < 0).astype(int)
            _wx_all["wx_heat_stress_flag"]     = (_wx_all["wx_temperature_2m_max"] > 35).astype(int)
            _wx_all["wx_extreme_precip_flag"]  = (_wx_all["wx_precipitation_sum"] > 25).astype(int)
            _wx_all["wx_extreme_wind_flag"]    = (_wx_all["wx_windspeed_10m_max"] > 50).astype(int)
            _wx_all["wx_high_humidity_flag"]   = (_wx_all["wx_relative_humidity_2m_max"] > 90).astype(int)

            # Rolling weather features (per facility, sorted by date)
            _wx_all = _wx_all.sort_values([_facility_col, "event_date"])
            _wx_all["wx_precip_roll3d"]     = (_wx_all.groupby(_facility_col)["wx_precipitation_sum"]
                                                .transform(lambda x: x.rolling(3, min_periods=1).sum()))
            _wx_all["wx_precip_roll7d"]     = (_wx_all.groupby(_facility_col)["wx_precipitation_sum"]
                                                .transform(lambda x: x.rolling(7, min_periods=1).sum()))
            _wx_all["wx_temp_mean_roll7d"]  = (_wx_all.groupby(_facility_col)["wx_temperature_2m_mean"]
                                                .transform(lambda x: x.rolling(7, min_periods=1).mean()))
            _wx_all["wx_freeze_thaw_roll7d"] = (_wx_all.groupby(_facility_col)["wx_freeze_flag"]
                                                .transform(lambda x: x.rolling(7, min_periods=1).sum()))

            # -- 5. Merge into main dataframe --------------------------------
            _before_cols = df.shape[1]
            df = df.merge(_wx_all, on=[_facility_col, "event_date"], how="left")
            _wx_cols = [c for c in df.columns if c.startswith("wx_")]
            _missing = df[_wx_cols].isnull().mean().max()

            print(f"[WEATHER] Fetched {len(_facilities)} facilities  "
                  f"({len(_seen_coords)} unique location(s))")
            print(f"[WEATHER] Weather cols added : {len(_wx_cols)}")
            print(f"[WEATHER] df shape after merge : {df.shape}  (was {_before_cols} cols)")
            print(f"[WEATHER] Max missing rate     : {_missing:.1%}")
            print(f"[WEATHER] Sample cols: {_wx_cols[:5]}")
            if _CITY_WEATHER_ONLY and _CITY_COL in df.columns:
                df = df.drop(columns=[_CITY_COL])
            WEATHER_ADDED = True

else:
    print("[WEATHER] Checkpoint loaded -- weather data already in df (if enriched before checkpoint).")


In [ ]:
# ── CELL 8 : Feature engineering (VALIDATOR) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["will_fail_3d"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#43A047","#E53935"], edgecolor="white")
axes[0].set_title("Target: will_fail_3d")

(1 - df.isnull().mean()).sort_values().head(20).plot(
    kind="barh", ax=axes[1], color="#1565C0")
axes[1].set_title("Feature completeness (lowest 20)")

df.groupby("event_date")["will_fail_3d"].sum().plot(
    ax=axes[2], color="#E53935", linewidth=0.8)
axes[2].set_title("Daily failure count over time")

plt.tight_layout()
plt.savefig("/tmp/ps1_eda.png", dpi=100)
plt.show()
print("EDA saved → /tmp/ps1_eda.png")

In [ ]:
# ── CELL 9 : Per-category temporal splits & resource config ─────────────────────
TARGET   = f"will_fail_{TARGET_HORIZON}"

# R3-2: Exclude VALIDATOR
df = df[df["mars_device_category"] == "VALIDATOR"].copy()
print(f"After VALIDATOR exclusion: {df.shape}  ({df['mars_device_category'].value_counts().to_dict()})")

# Sort once — all time-windowed features below depend on this

# ?? Align Gold Parquet column names to notebook expectations ??????????????
# Gold Parquet uses transit_day / event_count / oos_event_count;
# feature blocks below expect event_date / daily_event_count / oos_flag.
if 'transit_day' in df.columns and 'event_date' not in df.columns:
    df = df.rename(columns={'transit_day': 'event_date'})
if 'event_count' in df.columns and 'daily_event_count' not in df.columns:
    df = df.rename(columns={'event_count': 'daily_event_count'})
if 'oos_event_count' in df.columns and 'oos_flag' not in df.columns:
    df['oos_flag'] = (df['oos_event_count'] > 0).astype(int)
    print(f'Derived oos_flag from oos_event_count (mean={df["oos_flag"].mean():.3f})')

df = df.sort_values(["DEVICE_ID", "event_date"]).reset_index(drop=True)

ID_COLS  = ["DEVICE_ID", "event_date", "split"]

# ── Cast decimal.Decimal columns to float64 ───────────────────────────────────
# Delta Lake / Databricks exports DECIMAL columns as Python decimal.Decimal objects.
# pandas cannot do arithmetic between decimal.Decimal and float — cast everything up front.
import decimal as _decimal
_decimal_cols = [
    col for col in df.columns
    if df[col].dtype == object
    and len(df[col].dropna()) > 0
    and isinstance(df[col].dropna().iloc[0], _decimal.Decimal)
]
if _decimal_cols:
    df[_decimal_cols] = df[_decimal_cols].apply(pd.to_numeric, errors='coerce')
    print(f"Cast {len(_decimal_cols)} decimal.Decimal column(s) to float64: {_decimal_cols}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A — Rolling aggregates (original)
# ══════════════════════════════════════════════════════════════════════════════
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    if col in df.columns:
        df[f"{col}_roll{window}"] = (
            df.groupby("DEVICE_ID")[col]
              .transform(lambda x: x.rolling(window, min_periods=1).sum()))

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = (
        df.groupby("DEVICE_ID")["daily_event_count"]
          .transform(lambda x: (x - x.mean()) / (x.std() + 1e-6)))

if "use_revenue_decline_flag" in df.columns:
    df["use_revenue_decline_flag"] = df["use_revenue_decline_flag"].astype(int)

if "use_revenue_cents" in df.columns:
    df["use_revenue_cents_roll7"] = (
        df.groupby("DEVICE_ID")["use_revenue_cents"]
          .transform(lambda x: x.rolling(7, min_periods=1).mean()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B — Lag features t-1 / t-2 / t-3  (shift within device, no leakage)
# ══════════════════════════════════════════════════════════════════════════════
LAG_COLS = [c for c in ["oos_flag", "daily_event_count", "use_revenue_cents"] if c in df.columns]
for col in LAG_COLS:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df.groupby("DEVICE_ID")[col].shift(lag)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK C — Historical OOS counts + days-since + lifetime  (all shifted)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    for window in [30, 90]:
        df[f"oos_count_{window}d"] = (
            df.groupby("DEVICE_ID")["oos_flag"]
              .transform(lambda x, w=window: x.shift(1).rolling(w, min_periods=1).sum()))

    def _days_since_last_oos(x):
        cs = x.shift(1).cumsum()
        result = cs.groupby(cs).cumcount().astype(float)
        result[cs == 0] = np.nan
        return result

    df["days_since_last_oos"] = (
        df.groupby("DEVICE_ID")["oos_flag"].transform(_days_since_last_oos))

    df["oos_lifetime_count"] = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).cumsum()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D — Device age
# ══════════════════════════════════════════════════════════════════════════════
df["device_age_days"] = df.groupby("DEVICE_ID")["event_date"].transform(
    lambda x: (x - x.min()).dt.days)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E — Consecutive OOS streak  (acute deterioration signal)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    def _consec_oos(x):
        s = x.shift(1).fillna(0)
        grp = (s != s.shift(1)).cumsum()
        streak = s.groupby(grp).cumcount() + 1
        streak[s == 0] = 0.0
        return streak.astype(float)

    df["consec_oos_days"] = df.groupby("DEVICE_ID")["oos_flag"].transform(_consec_oos)
    df["max_streak_30d"]  = (
        df.groupby("DEVICE_ID")["consec_oos_days"]
          .transform(lambda x: x.shift(1).rolling(30, min_periods=1).max()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F — OOS rate acceleration  (trend: worsening or recovering?)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    _oos_7d_recent = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).sum())
    _oos_7d_prev   = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(8).rolling(7, min_periods=1).sum())
    df["oos_rate_accel_7d"] = (_oos_7d_recent - _oos_7d_prev).fillna(0)

    _oos_30d_recent = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).sum())
    _oos_30d_prev   = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(31).rolling(30, min_periods=1).sum())
    df["oos_rate_accel_30d"] = (_oos_30d_recent - _oos_30d_prev).fillna(0)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G — Exponentially weighted OOS  (recent OOS weighted higher than old)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    df["oos_ewm7"]  = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).ewm(span=7,  min_periods=1).mean()))
    df["oos_ewm30"] = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).ewm(span=30, min_periods=1).mean()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H — Recovery count  (OOS→OK transitions = recurrence-prone devices)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    _oos_lag1 = df.groupby("DEVICE_ID")["oos_flag"].shift(1).fillna(0)
    _oos_lag2 = df.groupby("DEVICE_ID")["oos_flag"].shift(2).fillna(0)
    df["_recovery"] = ((_oos_lag2 == 1) & (_oos_lag1 == 0)).astype(float)
    df["recovery_count_30d"] = (
        df.groupby("DEVICE_ID")["_recovery"]
          .transform(lambda x: x.rolling(30, min_periods=1).sum()))
    df["recovery_count_90d"] = (
        df.groupby("DEVICE_ID")["_recovery"]
          .transform(lambda x: x.rolling(90, min_periods=1).sum()))
    df = df.drop(columns=["_recovery"])

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I — Anomaly z-scores  (deviation from device's own rolling baseline)
# ══════════════════════════════════════════════════════════════════════════════
if "daily_event_count" in df.columns:
    _mu  = df.groupby("DEVICE_ID")["daily_event_count"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    _std = df.groupby("DEVICE_ID")["daily_event_count"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["event_count_zscore_30d"] = ((df["daily_event_count"] - _mu) / (_std + 1e-6)).fillna(0)

if "use_revenue_cents" in df.columns:
    _mu  = df.groupby("DEVICE_ID")["use_revenue_cents"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    _std = df.groupby("DEVICE_ID")["use_revenue_cents"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["revenue_zscore_30d"] = ((df["use_revenue_cents"].astype(float) - _mu) / (_std + 1e-6)).fillna(0)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J — Calendar features
# ══════════════════════════════════════════════════════════════════════════════
df["day_of_week"]  = df["event_date"].dt.dayofweek
df["month"]        = df["event_date"].dt.month
df["day_of_month"] = df["event_date"].dt.day
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["quarter"]      = df["event_date"].dt.quarter

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K — Peer comparison  (facility/location-level failure rate)
# ══════════════════════════════════════════════════════════════════════════════
_facility_col = next((c for c in df.columns
                      if any(k in c.lower() for k in ["facility", "station", "location", "site"])), None)
if _facility_col and "oos_flag" in df.columns:
    _peer_oos = (df.groupby(["event_date", _facility_col])["oos_flag"]
                   .transform("mean"))
    df["peer_oos_rate"] = _peer_oos
    df["oos_vs_peer"]   = df["oos_flag"] - _peer_oos

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK L — Interaction features
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns and "daily_event_count" in df.columns:
    df["oos_x_event_count"] = (
        df.groupby("DEVICE_ID")["oos_flag"].shift(1).fillna(0)
        * df.groupby("DEVICE_ID")["daily_event_count"].shift(1).fillna(0))

if "oos_ewm7" in df.columns and "device_age_days" in df.columns:
    df["oos_ewm7_x_age"] = df["oos_ewm7"] * np.log1p(df["device_age_days"])

# ══════════════════════════════════════════════════════════════════════════════
# LABEL LEAK EXCLUSION + FEATURE_COLS
# ══════════════════════════════════════════════════════════════════════════════

# Cast pre-computed 7-day decimal columns to float64 so they enter FEATURE_COLS
# (pandas excludes decimal(N,M) from is_numeric_dtype)
for _dc in ['availability_pct_7d', 'outage_min_7d']:
    if _dc in df.columns:
        df[_dc] = pd.to_numeric(df[_dc], errors='coerce').astype(float)
        print(f'  Cast {_dc} to float64')

LABEL_LEAK_COLS = [
    "failure_level",
    "max_failure_level",     # renamed from failure_level in Gold schema
    "is_chargeable",
    "is_hardware_oos",
    "is_hardware_event",
    "duration_min",
    "oos_duration_min",
    "will_fail_7d",
    "will_fail_14d",
    "will_fail_3d",   # exclude all horizons — active TARGET excluded separately
]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET] + LABEL_LEAK_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]

print(f"\n{'─'*60}")
print(f"Total features : {len(FEATURE_COLS)}")
blocks = {
    "A  rolling aggregates"  : [c for c in FEATURE_COLS if "_roll" in c or "zscore" == c.split("_")[-1]],
    "B  lag t-1/2/3"         : [c for c in FEATURE_COLS if "_lag" in c],
    "C  OOS history"         : [c for c in FEATURE_COLS if "oos_count" in c or "days_since" in c or "lifetime" in c],
    "D  device age"          : [c for c in FEATURE_COLS if "device_age" in c],
    "E  streak"              : [c for c in FEATURE_COLS if "streak" in c or "consec" in c],
    "F  acceleration"        : [c for c in FEATURE_COLS if "accel" in c],
    "G  EWM"                 : [c for c in FEATURE_COLS if "ewm" in c and "x_" not in c],
    "H  recovery"            : [c for c in FEATURE_COLS if "recovery" in c],
    "I  anomaly z-scores"    : [c for c in FEATURE_COLS if "zscore" in c],
    "J  calendar"            : [c for c in FEATURE_COLS if c in ["day_of_week","month","day_of_month","is_weekend","quarter"]],
    "K  peer"                : [c for c in FEATURE_COLS if "peer" in c or "vs_peer" in c],
    "L  interactions"        : [c for c in FEATURE_COLS if "_x_" in c],
}
for name, cols in blocks.items():
    if cols:
        print(f"  Block {name:<28}: {len(cols):2d} features")
print(f"{'─'*60}")
print(f"Leak cols excluded: {[c for c in LABEL_LEAK_COLS if c in df.columns]}")
print("Feature engineering complete — run CELL 6b → CELL 10 → CELLs 11-23")


# ── BLOCK F — Device-level rolling failure rate (ported from PS1_Evaluation_Fix) ──
# Lagged 1 day to prevent leakage; 30-day rolling mean per device.
_rate_src = "will_fail_3d_clean" if "will_fail_3d_clean" in df.columns else TARGET
df = df.sort_values(["DEVICE_ID", "event_date"]).copy()
df["device_fail_rate_30d"] = (
    df.groupby("DEVICE_ID")[_rate_src]
    .transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean().fillna(0))
)
if "device_fail_rate_30d" not in FEATURE_COLS:
    FEATURE_COLS.append("device_fail_rate_30d")
print(f"  device_fail_rate_30d added  (mean={df['device_fail_rate_30d'].mean():.4f})")

# ── BLOCK G — Anomaly × failure-count interaction features ────────────────────
# Cross-signal: devices with both anomaly scores AND recent failure history are
# highest risk. Interaction captures the multiplicative effect.
if "if_anomaly_score" in df.columns:
    _fail_roll_cols = sorted([
        c for c in df.columns
        if "fail" in c.lower() and any(s in c for s in ["7d", "14d", "30d"])
    ])
    _inter_added = []
    for _fc in _fail_roll_cols[:3]:   # cap at 3 to avoid feature explosion
        _ic = f"anomaly_x_{_fc}"
        df[_ic] = df["if_anomaly_score"] * df[_fc]
        if _ic not in FEATURE_COLS:
            FEATURE_COLS.append(_ic)
        _inter_added.append(_ic)
    print(f"  Anomaly interaction features added: {_inter_added}")
else:
    print("  [Skip] if_anomaly_score not in df — no interaction features")

In [ ]:
# -- CELL 10 : Clean label -- exclude planned maintenance from will_fail_3d (VALIDATOR) -
# VALIDATOR label (R6-1 2026-07-17) = OOS Set events from device_failures.
# SQL-level: device_failures already filters is_oos_event=TRUE AND EVENT_STATE_TYPE_NAME='Set'.
# Notebook-level: additionally exclude known planned maintenance event_type_ids if the
# column is present in the gold table (event_type_id from device_event_enriched).

TARGET = "will_fail_3d"
if "event_type_id" in df.columns and len(VALIDATOR_PLANNED_EVENT_TYPES) > 0:
    planned_mask = df["event_type_id"].isin(VALIDATOR_PLANNED_EVENT_TYPES)
    df["will_fail_3d_clean"] = df[TARGET].where(~planned_mask, 0)
    n_excl = planned_mask.sum()
    pct    = n_excl / max(len(df), 1) * 100
    print(f"Planned maintenance (event_type_id) excluded: {n_excl:,} rows ({pct:.2f}%) -> label forced to 0")
    TARGET = "will_fail_3d_clean"
elif "is_planned_maintenance" in df.columns:
    df["will_fail_3d_clean"] = df[TARGET].where(~df["is_planned_maintenance"].fillna(False), 0)
    n_excl = df["is_planned_maintenance"].sum()
    print(f"Planned maintenance (is_planned_maintenance flag) excluded: {n_excl:,} rows")
    TARGET = "will_fail_3d_clean"
else:
    print("No planned-maintenance column found in gold table.")
    print("VALIDATOR label pre-filtered at SQL level (device_failures OOS Set events only).")
print(f"Label column in use: {TARGET}")
print()

# TVM and GATE trained as SEPARATE models.
# Each gets its own: split, scale_pos_weight, MLflow experiment,
# Feature Store group, Model Registry, and SageMaker endpoint.

CATEGORIES = {}
for cat in ["VALIDATOR"]:
    cat_df = df[df["mars_device_category"] == cat].sort_values(["DEVICE_ID", "event_date"])

    tr = cat_df[cat_df["event_date"] <  "2025-10-01"]
    va = cat_df[(cat_df["event_date"] >= "2025-10-01") & (cat_df["event_date"] < "2026-01-01")]
    te = cat_df[cat_df["event_date"] >= "2026-01-01"]

    n0, n1 = (tr[TARGET] == 0).sum(), (tr[TARGET] == 1).sum()

    CATEGORIES[cat] = {
        "train_df": tr, "val_df": va, "test_df": te,
        "X_train": tr[FEATURE_COLS].values, "y_train": tr[TARGET].values,
        "X_val":   va[FEATURE_COLS].values, "y_val":   va[TARGET].values,
        "X_test":  te[FEATURE_COLS].values, "y_test":  te[TARGET].values,
        "X_cv":    np.vstack([tr[FEATURE_COLS].values, va[FEATURE_COLS].values]),
        "y_cv":    np.concatenate([tr[TARGET].values, va[TARGET].values]),
        "scale_pos_weight": round(n0 / max(n1, 1), 2),
        "neg": n0, "pos": n1,
        "experiment_name": f"chicago-ps1-3d-{cat.lower()}-failure",
        "model_name":      f"ps1-3d-{cat.lower()}-device-failure",
        "registry_name":   f"chicago-ps1-3d-{cat.lower()}-failure",
        "fs_group":        f"chicago-ps1-3d-{cat.lower()}-features",
        "endpoint":        f"chicago-ps1-3d-{cat.lower()}-failure-v1",
        # Recall floor for threshold tuning on val PR curve.
        # GATE: only 0.60% positive rate (scale_pos_weight ~166) —
        # recall=0.80 may not be achievable; use 0.70 as the floor.
        "recall_floor": 0.80 if cat == "TVM" else 0.70,
    }
    mlflow.set_experiment(CATEGORIES[cat]["experiment_name"])

    print(f"\n── {cat} ──────────────────────────────────────────────────────")
    print(f"  Train  : {len(tr):>8,}  ({tr['event_date'].min().date()} -> {tr['event_date'].max().date()})")
    print(f"  Val    : {len(va):>8,}")
    print(f"  Test   : {len(te):>8,}")
    print(f"  Pos rate (train): {n1/max(len(tr),1):.2%}  ({n1:,} positives)")
    print(f"  scale_pos_weight: {CATEGORIES[cat]['scale_pos_weight']}")
    print(f"  MLflow experiment: {CATEGORIES[cat]['experiment_name']}")

# ── Save checkpoint after per-category splits ────────────────────────────
import os, joblib as _jl
_ckpt_dir = "/home/sagemaker-user/PS1V/checkpoints"
import shutil as _shutil
_shutil.rmtree(_ckpt_dir, ignore_errors=True)  # clear stale checkpoint
os.makedirs(_ckpt_dir, exist_ok=True)
df.to_parquet(os.path.join(_ckpt_dir, "df.parquet"), index=True)
# Save scalar/string metadata per category (no DataFrames or arrays)
_scalar_keys = ["scale_pos_weight","neg","pos","recall_floor",
                "experiment_name","model_name","registry_name",
                "fs_group","endpoint"]
_jl.dump({
    "FEATURE_COLS":    FEATURE_COLS,
    "TARGET":          TARGET,
    "ID_COLS":         ID_COLS,
    "CATEGORIES_meta": {
        cat: {k: v for k, v in data.items() if k in _scalar_keys}
        for cat, data in CATEGORIES.items()
    },
}, os.path.join(_ckpt_dir, "meta.joblib"))
# Save numpy arrays per category separately (can be large)
_jl.dump({
    cat: {k: data[k] for k in ["X_train","y_train","X_val","y_val",
                                 "X_test","y_test","X_cv","y_cv"]}
    for cat, data in CATEGORIES.items()
}, os.path.join(_ckpt_dir, "categories_arrays.joblib"))
print(f"[CHECKPOINT] Saved to {_ckpt_dir}")
for cat, data in CATEGORIES.items():
    print(f"  {cat}: X_train={data['X_train'].shape}  pos={data['pos']:,}")
print(f"  Next kernel restart: run Cells 1-3-3b only, then jump to Cell 9.")


In [ ]:
# ── CELL 11 : Feature Store — create group + seed ONLINE store ──────────────────
# Planned events (software resets, depot maintenance, scheduled PM) inflate the
# positive rate and teach the model to predict operator-initiated actions rather
# than unplanned failures. Recompute will_fail_3d_clean when event_type_id available.
# Ported from PS1_Evaluation_Fix.ipynb Cell 4b.

_USE_CLEAN_LABEL = False

if "event_type_id" in df.columns:
    print("event_type_id found — recomputing will_fail_3d_clean ...")
    _planned = {
        "TVM":  set(TVM_PLANNED_EVENT_TYPES),
        "GATE": set(GATE_PLANNED_EVENT_TYPES),
    }
    df["_unplanned"] = 1

    for _cat in ["VALIDATOR"]:
        _mask = df["mars_device_category"] == _cat
        df.loc[_mask & df["event_type_id"].isin(_planned[_cat]), "_unplanned"] = 0
        _zeroed = int((_mask & (df["_unplanned"] == 0)).sum())
        print(f"  {_cat:<6}: {_zeroed:,} planned rows zeroed ({len(_planned[_cat])} event types)")

    df = df.sort_values(["DEVICE_ID", "event_date"]).copy()
    df["will_fail_3d_clean"] = (
        df.groupby("DEVICE_ID")["_unplanned"]
        .transform(lambda x: x.shift(-1).rolling(3, min_periods=1).max().fillna(0))
        .astype(int)
    )
    df.drop(columns=["_unplanned"], inplace=True)
    _USE_CLEAN_LABEL = True

    print("\n[Clean label positive rate]")
    for _cat in ["VALIDATOR"]:
        _m = df[df["mars_device_category"] == _cat]
        _orig  = _m[TARGET].mean()
        _clean = _m["will_fail_3d_clean"].mean()
        print(f"  {_cat:<6}: original={_orig:.3%}  clean={_clean:.3%}  "
              f"delta={_orig - _clean:.3%}  (planned events excluded)")

    # Update CATEGORIES splits to use clean label
    _target_col = "will_fail_3d_clean"
    for _cat in ["VALIDATOR"]:
        for _split in ["train_df", "val_df", "test_df"]:
            _df_s = CATEGORIES[_cat][_split]
            if _target_col in _df_s.columns:
                CATEGORIES[_cat][f"y_{_split.replace('_df','')}_clean"] = _df_s[_target_col].values
    print("\n[ACTIVE] will_fail_3d_clean applied to CATEGORIES splits.")
else:
    print("[Skip] event_type_id not available — using original will_fail_3d label.")
    print(f"  TVM  planned types ({len(TVM_PLANNED_EVENT_TYPES)}): {TVM_PLANNED_EVENT_TYPES}")
    print(f"  GATE planned types ({len(GATE_PLANNED_EVENT_TYPES)}): {GATE_PLANNED_EVENT_TYPES}")

In [ ]:
# ── CELL 12 : Feature Store — spot-check online store per category ──────────────
if not ENABLE_FEATURE_STORE:
    print('Feature Store disabled -- skipping CELL 7.')
else:
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
    import time

    OFFLINE_URI = f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}"

    # Latest record per device (full df -- split not yet assigned in CELL 9)
    fs_cols = [c for c in FEATURE_COLS + [TARGET, 'DEVICE_ID', 'event_date'] if c in df.columns]
    fs_full = df[fs_cols].copy()
    fs_full['DEVICE_ID'] = fs_full['DEVICE_ID'].astype(str)
    fs_latest = (fs_full.sort_values('event_date')
                         .groupby('DEVICE_ID', as_index=False).tail(1)
                         .reset_index(drop=True))
    fs_latest['event_time'] = fs_latest['event_date'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    fs_latest = fs_latest.drop(columns=['event_date'])
    num_cols = fs_latest.select_dtypes(include='number').columns
    fs_latest[num_cols] = fs_latest[num_cols].fillna(0)
    print(f'Candidate columns for Feature Store: {len(fs_latest.columns)}')

    def fs_type(dt):
        if pd.api.types.is_float_dtype(dt):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dt): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    def _make_fg(cols_df):
        return FeatureGroup(
            name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
            feature_definitions=[FeatureDefinition(c, fs_type(cols_df[c].dtype))
                                 for c in cols_df.columns])

    def _fg_status(fg):
        try: return fg.describe().get('FeatureGroupStatus')
        except Exception: return None

    def _fg_registered_cols(fg):
        try: return {f['FeatureName'] for f in fg.describe().get('FeatureDefinitions', [])}
        except Exception: return set()

    # ?? Delete stale Feature Group if schema changed ??????????????????????????
    _probe = _make_fg(fs_latest)
    st = _fg_status(_probe)
    if st in ('Created', 'CreateFailed'):
        existing = _fg_registered_cols(_probe)
        if st == 'CreateFailed' or (set(fs_latest.columns) - existing):
            print(f'Deleting stale Feature Group (status={st})...')
            _probe.delete()
            while True:
                try: _make_fg(fs_latest).describe(); print('  Still deleting...'); time.sleep(10)
                except Exception: print('  Deleted.'); break
            st = None
        else:
            print(f'Schema OK ({len(existing)} cols) -- reusing existing Feature Group.')

    # ?? Create fresh Feature Group ????????????????????????????????????????????
    if st is None:
        feature_group = _make_fg(fs_latest)
        feature_group.create(
            s3_uri=OFFLINE_URI, record_identifier_name='DEVICE_ID',
            event_time_feature_name='event_time', role_arn=ROLE,
            enable_online_store=True,
            description='PS1 daily device features -- Chicago Ventra')
        print('Creating Feature Group...')
        while _fg_status(feature_group) != 'Created':
            time.sleep(15); print('  status:', _fg_status(feature_group))
        print('Waiting 60s for schema propagation...')
        time.sleep(60)
    else:
        feature_group = _probe

    # ?? Verify what actually got registered (SDK may drop engineered cols) ???
    registered = _fg_registered_cols(feature_group)
    unregistered = set(fs_latest.columns) - registered
    if unregistered:
        print(f'NOTE: {len(unregistered)} columns not registered by SDK (likely in-memory only):')
        print(f'  {sorted(unregistered)}')
        print(f'Ingesting {len(registered)} registered columns only.')
        fs_ingest = fs_latest[[c for c in fs_latest.columns if c in registered]].copy()
    else:
        fs_ingest = fs_latest
        print(f'All {len(registered)} columns registered -- ingesting full set.')

    print(f'Feature Group ready: {FS_GROUP_NAME}  ({len(registered)} features)')

    # ?? Seed online store + verify ????????????????????????????????????????????
    feature_group.ingest(data_frame=fs_ingest.reset_index(drop=True), max_workers=4, wait=True)
    sample_id = fs_ingest['DEVICE_ID'].iloc[0]
    rec = feature_group.get_record(record_identifier_value_as_string=sample_id)
    print(f'Ingested {len(fs_ingest):,} records | online sample ({sample_id}):',
          {f['FeatureName']: f['ValueAsString'] for f in (rec or [])[:4]})
    globals()['feature_group'] = feature_group


In [ ]:
# ── CELL 13 : Helper functions (evaluation + threshold + ranking + quality gate) ───
# Creation and ingestion are in CELL 7. Verifies one TVM + one GATE device.
if not ENABLE_FEATURE_STORE:
    print("Feature Store disabled — skipping CELL 8.")
else:
    for cat_name in ["VALIDATOR"]:
        sample_ids = (df[df["mars_device_category"] == cat_name]["DEVICE_ID"]
                      .astype(str).unique()[:1])
        if len(sample_ids):
            rec = feature_group.get_record(record_identifier_value_as_string=sample_ids[0])
            print(f"[{cat_name}] online store ({sample_ids[0]}):",
                  {f['FeatureName']: f['ValueAsString'] for f in (rec or [])[:4]})
        else:
            print(f"[{cat_name}] no devices found for spot-check")
    print("Feature Store verification complete.")


In [ ]:
# ── CELL 14 : SMOTE checkpoint — load post-imputation/SMOTE arrays ──────────────
# X_train/val/test, y_train/val/test are set at the top of each CATEGORY loop.

def make_common_tags(category):
    return {
        "ps"           : "PS1",
        "category"     : category,
        "target"       : TARGET,
        "city"         : "chicago",
        "gold_version" : str(GOLD_VERSION),
        "grain"        : "device_day",
        "architecture" : "CUBIC-MARS-AWS-v6",
    }


def evaluate_all(model, name):
    """Train / val / test evaluation in one call.

    Displays inline:
      1. Metrics summary table  (AUC / AP / Acc / F1 / Prec / Rec x train/val/test)
      2. Confusion matrices     (3-panel: train / val / test)
      3. ROC + Precision-Recall curves  (3 coloured lines each)
      4. Classification reports (all three splits, printed + logged)

    Logs all artefacts to the active MLflow run.
    Returns a flat dict of all split metrics for mlflow.log_metrics().
    """
    from sklearn.metrics import accuracy_score as _acc_fn
    splits = [('train', X_train, y_train),
               ('val',   X_val,   y_val),
               ('test',  X_test,  y_test)]
    all_m  = {}
    probs  = {}

    # ?? 1. scalar metrics (all three splits) ??????????????????????????????
    for split, X, y in splits:
        prob = model.predict_proba(X)[:, 1]
        pred = (prob >= 0.5).astype(int)
        probs[split] = prob
        all_m[f'{split}_auc']  = round(roc_auc_score(y, prob), 4)
        all_m[f'{split}_ap']   = round(average_precision_score(y, prob), 4)
        all_m[f'{split}_acc']  = round(_acc_fn(y, pred), 4)
        all_m[f'{split}_f1']   = round(f1_score(y, pred), 4)
        all_m[f'{split}_prec'] = round(precision_score(y, pred, zero_division=0), 4)
        all_m[f'{split}_rec']  = round(recall_score(y, pred, zero_division=0), 4)

    # ?? 2. metrics summary table ???????????????????????????????????????????
    _H = f'  {"Split":<6}  {"AUC":>7}  {"AP":>7}  {"Acc":>7}  {"F1":>7}  {"Prec":>7}  {"Rec":>7}'
    _S = f'  {"-"*6}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}'
    print(f'\n{name}')
    print(_H); print(_S)
    for _s, _, _ in splits:
        print(f'  {_s:<6}  '
              f'{all_m[_s+"_auc"]:>7.4f}  '
              f'{all_m[_s+"_ap"]:>7.4f}  '
              f'{all_m[_s+"_acc"]:>7.4f}  '
              f'{all_m[_s+"_f1"]:>7.4f}  '
              f'{all_m[_s+"_prec"]:>7.4f}  '
              f'{all_m[_s+"_rec"]:>7.4f}')

    # ?? 3. confusion matrices (3-panel) ???????????????????????????????????
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{name} -- Confusion Matrices', fontsize=12, fontweight='bold')
    for ax, (split, X, y) in zip(axes, splits):
        prob = probs[split]
        pred = (prob >= 0.5).astype(int)
        cm   = confusion_matrix(y, pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['No Fail', 'Fail'], yticklabels=['No Fail', 'Fail'])
        ax.set_title(
            f'{split.upper()}  AUC={all_m[split+"_auc"]:.3f}  Acc={all_m[split+"_acc"]:.3f}\n'
            f'F1={all_m[split+"_f1"]:.3f}  Prec={all_m[split+"_prec"]:.3f}  Rec={all_m[split+"_rec"]:.3f}')
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    _p = f'/tmp/cm_{name}.png'
    fig.savefig(_p, bbox_inches='tight', dpi=100); plt.show(); plt.close(fig)
    mlflow.log_artifact(_p)

    # ?? 4. ROC + Precision-Recall curves ??????????????????????????????????
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    _clr = {'train': '#1565C0', 'val': '#E65100', 'test': '#2E7D32'}
    for split, X, y in splits:
        prob        = probs[split]
        fpr, tpr, _ = roc_curve(y, prob)
        pr, rc, _   = precision_recall_curve(y, prob)
        col         = _clr[split]
        axes[0].plot(fpr, tpr, color=col, lw=1.6,
                     label=f'{split}  AUC={all_m[split+"_auc"]:.4f}')
        axes[1].plot(rc, pr, color=col, lw=1.6,
                     label=f'{split}  AP={all_m[split+"_ap"]:.4f}')
    axes[0].plot([0,1],[0,1], 'k--', lw=0.7)
    axes[0].set(xlabel='FPR', ylabel='TPR', title=f'{name} -- ROC Curve')
    axes[0].legend(fontsize=9)
    axes[1].set(xlabel='Recall', ylabel='Precision', title=f'{name} -- Precision-Recall')
    axes[1].legend(fontsize=9)
    plt.tight_layout()
    _p = f'/tmp/roc_pr_{name}.png'
    fig.savefig(_p, bbox_inches='tight', dpi=100); plt.show(); plt.close(fig)
    mlflow.log_artifact(_p)

    # ?? 5. classification reports (train / val / test) ?????????????????????
    print(f'\n{"="*66}')
    print(f'  {name} -- Classification Reports')
    print(f'{"="*66}')
    for split, X, y in splits:
        prob = probs[split]
        pred = (prob >= 0.5).astype(int)
        rpt  = classification_report(y, pred,
                                     target_names=['No Fail', 'Fail'], zero_division=0)
        print(f'\n  {split.upper()} set'
              f'  (AUC={all_m[split+"_auc"]:.4f}  Acc={all_m[split+"_acc"]:.4f}):')
        print(rpt)
        _p = f'/tmp/clf_rpt_{name}_{split}.txt'
        with open(_p, 'w') as _f:
            _f.write(f'{name} -- {split}\n{rpt}')
        mlflow.log_artifact(_p)

    return all_m


def log_feat_importance(importances, name, top_n=25):
    """Top-N feature importance bar chart -- shown inline and logged to MLflow."""
    if importances is None or len(importances) == 0:
        return
    n  = min(len(importances), len(FEATURE_COLS))
    fi = (pd.Series(importances[:n], index=FEATURE_COLS[:n])
            .sort_values(ascending=False).head(top_n))
    fig, ax = plt.subplots(figsize=(9, max(4, len(fi) * 0.35)))
    fi.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.invert_yaxis()
    ax.set_title(f'{name} -- Top {top_n} Feature Importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    _p = f'/tmp/feat_imp_{name}.png'
    fig.savefig(_p, bbox_inches='tight', dpi=100); plt.show(); plt.close(fig)
    mlflow.log_artifact(_p)
    _c = f'/tmp/feat_imp_{name}.csv'
    fi.reset_index().rename(columns={'index':'feature', 0:'importance'}).to_csv(_c, index=False)
    mlflow.log_artifact(_c)



# ── Helpers added from PS1_Evaluation_Fix.ipynb ──────────────────────────────

def threshold_sweep(y_true, y_prob, thresholds=None):
    """Precision/Recall/F1 table across a range of thresholds."""
    import pandas as _pd_ts
    if thresholds is None:
        thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        prec = tp / max(tp + fp, 1)
        rec  = tp / max(tp + fn, 1)
        f1   = 2 * prec * rec / max(prec + rec, 1e-9)
        rows.append({"threshold": t, "Accuracy": round((tp + (len(y_true)-tp-fp-fn)) / len(y_true), 4),
                     "Recall": round(rec, 4), "Precision": round(prec, 4), "F1": round(f1, 4),
                     "TP": tp, "FP": fp, "FN": fn})
    return _pd_ts.DataFrame(rows)


def print_ranking_table(label, y_true, y_prob, top_ks=(0.01, 0.05, 0.10)):
    """Precision@K / Recall@K / Lift@K for ranked dispatch evaluation."""
    import numpy as _np_rk
    base_rate = float(y_true.mean())
    _order    = _np_rk.argsort(y_prob)[::-1]
    print(f"\n{label} — Ranking metrics (base rate = {base_rate:.3%})")
    print(f"  {'K':<8}  {'Prec@K':>8}  {'Rec@K':>8}  {'Lift@K':>8}")
    print(f"  {'──────':<8}  {'────────':>8}  {'────────':>8}  {'────────':>8}")
    for k in top_ks:
        n_k   = max(int(len(y_true) * k), 1)
        top_y = y_true.values[_order[:n_k]] if hasattr(y_true, "values") else y_true[_order[:n_k]]
        prec  = top_y.mean()
        rec   = top_y.sum() / max(y_true.sum(), 1)
        lift  = prec / max(base_rate, 1e-9)
        print(f"  Top {k:.0%}  {prec:>8.4f}  {rec:>8.4f}  {lift:>7.2f}x")
    print(f"  [Ops target] Top 5% lift > 2.0 = model useful for ranked dispatch.")


def quality_gate_eval(category, model_or_proba, X_or_none, y_true, threshold, recall_floor):
    """Returns True if model passes recall and AP gates. Prints result."""
    import numpy as _np_qg
    from sklearn.metrics import average_precision_score as _aps
    if hasattr(model_or_proba, "predict_proba"):
        y_prob = model_or_proba.predict_proba(X_or_none)[:, 1]
    else:
        y_prob = model_or_proba
    pred      = (y_prob >= threshold).astype(int)
    tp        = int(((pred == 1) & (y_true == 1)).sum())
    fn        = int(((pred == 0) & (y_true == 1)).sum())
    rec       = tp / max(tp + fn, 1)
    ap        = float(_aps(y_true, y_prob))
    pos_rate  = float(y_true.mean())
    issues    = []
    if rec < recall_floor:
        issues.append(f"recall={rec:.3f} < floor={recall_floor:.2f}")
    if ap <= pos_rate:
        issues.append(f"AP={ap:.4f} <= baseline={pos_rate:.4f}")
    if issues:
        print(f"  [{category}] GATE FAIL: " + "; ".join(issues))
        return False
    print(f"  [{category}] GATE PASS: recall={rec:.3f}  AP={ap:.4f}  baseline={pos_rate:.4f}")
    return True

In [ ]:
# ── CELL 15 : Per-category setup — imputation + SMOTE + CATEGORY_RESULTS ────────
# If checkpoint exists: restores imputed+SMOTEd arrays into CATEGORIES and
# initialises CATEGORY_RESULTS → skip Cell 10 entirely (saves 30-90 min).
# Set FORCE_SMOTE=True to re-run imputation + SMOTE from scratch.
# Requires Cell 3b checkpoint to have loaded first (CATEGORIES must exist).

import os, joblib as _jl_sm

_SMOTE_CKPT  = "/home/sagemaker-user/PS1V/checkpoints/smote_arrays.joblib"
FORCE_SMOTE  = False
SMOTE_LOADED = False

if not FORCE_SMOTE and os.path.exists(_SMOTE_CKPT):
    print("[SMOTE CKPT] Loading post-imputation/SMOTE arrays ...")
    _smote_data = _jl_sm.load(_SMOTE_CKPT)
    CATEGORY_RESULTS = {cat: {} for cat in ["VALIDATOR"]}
    for cat, arrs in _smote_data.items():
        CATEGORIES[cat]["X_train"]          = arrs["X_train"]
        CATEGORIES[cat]["y_train"]          = arrs["y_train"]
        CATEGORIES[cat]["X_val"]            = arrs["X_val"]
        CATEGORIES[cat]["X_test"]           = arrs["X_test"]
        CATEGORIES[cat]["X_cv"]             = arrs["X_cv"]
        CATEGORIES[cat]["pos"]              = arrs["pos"]
        CATEGORIES[cat]["scale_pos_weight"] = arrs["scale_pos_weight"]
        CATEGORY_RESULTS[cat]["train_median"] = arrs["train_median"]
        smote_tag = " [SMOTE applied]" if cat == "GATE" else ""
        print(f"  {cat}: X_train={arrs['X_train'].shape}"
              f"  pos={arrs['pos']:,} ({arrs['pos']/len(arrs['y_train']):.2%}){smote_tag}")
    SMOTE_LOADED = True
    print()
    print("  SMOTE CHECKPOINT LOADED — skip Cell 10.")
    print("  Jump to Cell 11 (MODEL 1 — Logistic Regression).")
else:
    print("[SMOTE CKPT] Not found or FORCE_SMOTE=True — run Cell 10 (imputation + SMOTE).")
    print("  Checkpoint will be saved automatically at end of Cell 10.")


In [ ]:
# ── CELL 16 : Rolling failure history features ──────────────────────────────────
# Run once before any training cell. CATEGORY_RESULTS accumulates across cells below.

# ?? Feature null-rate audit (train set, before imputation) ??????????????????
# Shows which FEATURE_COLS are effectively dead (>90% null) across each category.
# Dead columns survived the LABEL_LEAK_COLS filter but carry no real signal.
print('\n?? Feature null-rate audit (TRAIN set, before imputation) ??')
for _cat_name in ['VALIDATOR']:
    _tr = CATEGORIES[_cat_name]['train_df'][FEATURE_COLS]
    _nr = _tr.isnull().mean().sort_values(ascending=False)
    _dead   = _nr[_nr > 0.90]
    _sparse = _nr[(_nr > 0.10) & (_nr <= 0.90)]
    _full   = _nr[_nr <= 0.10]
    print(f'\n  {_cat_name}: {len(FEATURE_COLS)} features total')
    print(f'    Full   (<=10 % null): {len(_full):3d}')
    print(f'    Sparse (10-90% null): {len(_sparse):3d}')
    print(f'    Dead   (>90 % null): {len(_dead):3d}')
    if len(_dead):
        print(f'    Dead columns:')
        for _col in _dead.index:
            print(f'      {_col:45s}  {_dead[_col]:.1%} null')
    if len(_sparse):
        print(f'    Sparse columns (top 10):')
        for _col in _sparse.index[:10]:
            print(f'      {_col:45s}  {_sparse[_col]:.1%} null')
print()

CATEGORY_RESULTS = {cat: {} for cat in ['VALIDATOR']}

for CATEGORY in ['VALIDATOR']:
    cat = CATEGORIES[CATEGORY]

    # Cast to float64 before imputation (integer/object dtypes break nanmedian)
    for key in ('X_train', 'X_val', 'X_test', 'X_cv'):
        cat[key] = cat[key].astype(float)

    # Train-only median imputation — no leakage from val/test
    train_median = np.nanmedian(cat['X_train'], axis=0)
    train_median = np.where(np.isnan(train_median), 0.0, train_median)  # all-null cols → 0
    cat['X_train'] = np.where(np.isnan(cat['X_train']), train_median, cat['X_train'])
    cat['X_val']   = np.where(np.isnan(cat['X_val']),   train_median, cat['X_val'])
    cat['X_test']  = np.where(np.isnan(cat['X_test']),  train_median, cat['X_test'])
    cat['X_cv']    = np.where(np.isnan(cat['X_cv']),    train_median, cat['X_cv'])
    CATEGORY_RESULTS[CATEGORY]['train_median'] = train_median

    # SMOTE oversampling for GATE only (0.6% positive → ~5% after resampling)
    # Applied to X_train/y_train only — val/test stay original for unbiased eval.
    if CATEGORY == 'GATE':
        from imblearn.over_sampling import SMOTE
        # Store pre-SMOTE arrays for honest full-data test predictions in stacking
        CATEGORY_RESULTS[CATEGORY]['X_train_pre_smote'] = cat['X_train'].copy()
        CATEGORY_RESULTS[CATEGORY]['y_train_pre_smote'] = cat['y_train'].copy()
        _n_pos = int(cat['y_train'].sum())
        _n_neg = len(cat['y_train']) - _n_pos
        # Mirror imblearn's internal check: int(n_majority * ratio) - n_minority > 0
        _smote_new = int(_n_neg * SMOTE_RATIO) - _n_pos
        if _smote_new > 0:
            try:
                _sm = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=42)
                cat['X_train'], cat['y_train'] = _sm.fit_resample(cat['X_train'], cat['y_train'])
                print(f'  [{CATEGORY}] SMOTE: +{_smote_new} synthetic samples '
                      f'({_n_pos}/{_n_neg} -> {int(cat["y_train"].sum())}/{_n_neg})')
            except ValueError as _se:
                print(f'  [{CATEGORY}] SMOTE skipped (edge case): {_se}')
        else:
            print(f'  [{CATEGORY}] SMOTE skipped: {_n_pos}/{_n_neg} already at or above target '
                  f'({_n_pos/_n_neg:.4f} >= {SMOTE_RATIO})')
        cat['pos'] = int(cat['y_train'].sum())
        _n_smoted  = len(cat['y_train'])
        cat['scale_pos_weight'] = round((_n_smoted - cat['pos']) / cat['pos'], 2)
        # Rebuild X_cv with original (pre-SMOTE) train arrays — SMOTE is re-applied in stacking
        cat['X_cv'] = np.vstack([CATEGORY_RESULTS[CATEGORY]['X_train_pre_smote'], cat['X_val']])
        cat['y_cv'] = np.concatenate([CATEGORY_RESULTS[CATEGORY]['y_train_pre_smote'], cat['y_val']])

    n_total = len(cat['X_train'])
    print(f'\n── {CATEGORY} ──────────────────────────────────────────')
    print(f"  Train  : {n_total:,} rows  ({cat['pos']:,} pos = {cat['pos']/n_total:.2%})"
          + (" [SMOTE applied]" if CATEGORY == 'GATE' else ""))
    print(f"  Val    : {len(cat['X_val']):,}")
    print(f"  Test   : {len(cat['X_test']):,}")
    print(f"  scale_pos_weight : {cat['scale_pos_weight']}")
    print(f"  recall_floor     : {cat['recall_floor']}")
    print(f"  Experiment       : {cat['experiment_name']}")

print('\nSetup complete — run CELLs 11-23 in order.')

# ── Save post-imputation/SMOTE checkpoint ────────────────────────────────
import os, joblib as _jl_sm2
_smote_ckpt = "/home/sagemaker-user/PS1V/checkpoints/smote_arrays.joblib"
_jl_sm2.dump({
    cat: {
        "X_train":          CATEGORIES[cat]["X_train"],
        "y_train":          CATEGORIES[cat]["y_train"],
        "X_val":            CATEGORIES[cat]["X_val"],
        "X_test":           CATEGORIES[cat]["X_test"],
        "X_cv":             CATEGORIES[cat]["X_cv"],
        "pos":              CATEGORIES[cat]["pos"],
        "scale_pos_weight": CATEGORIES[cat]["scale_pos_weight"],
        "train_median":     CATEGORY_RESULTS[cat]["train_median"],
    }
    for cat in ["VALIDATOR"]
}, _smote_ckpt)
print(f"\n[SMOTE CKPT] Saved → {_smote_ckpt}")
for cat in ["VALIDATOR"]:
    x = CATEGORIES[cat]["X_train"]
    print(f"  {cat}: X_train={x.shape}  pos={CATEGORIES[cat]['pos']:,}"
          f"  ({CATEGORIES[cat]['pos']/len(CATEGORIES[cat]['y_train']):.2%})")
print("  Next restart: load Cell 3b + Cell 9b → jump to Cell 11.")


In [ ]:
# ── CELL 17 : MODEL 1 — Logistic Regression (per category) ──────────────────────
# Adds 5 engineered features capturing each VALIDATOR device's recent failure
# history. No leakage: rolling window covers only strictly prior dates
# (label shifted +1 within each device). Appended to FEATURE_COLS and
# X_train/val/test/cv arrays in-place.
# Run AFTER Cell 10 (imputation) and BEFORE Cell 11 (model training).
# Re-runnable: per-category shape guard skips if already extended.

import numpy as np, pandas as pd

_ROLL_NEW_COLS = [
    'roll_fail_7d',     # failures in past 7 days for this device
    'roll_fail_30d',    # failures in past 30 days
    'roll_fail_90d',    # failures in past 90 days
    'days_since_fail',  # days since most recent prior failure (999 = never)
    'fail_free_streak', # consecutive failure-free days before today
]

def _roll_feats_for_category(cat):
    # Auto-detect date + device columns
    _date_col = next((c for c in ['TRANSIT_DAY_KEY', 'event_date', 'date', 'DATE']
                      if c in cat['train_df'].columns), None)
    _dev_col  = next((c for c in ['DEVICE_KEY', 'device_key', 'DEVICE_ID', 'device_id']
                      if c in cat['train_df'].columns), None)
    if not _date_col or not _dev_col:
        raise ValueError(f"Cannot auto-detect cols. train_df cols: "
                         f"{list(cat['train_df'].columns[:25])}")

    # Build tagged combined dataframe (train + val + test)
    _parts = []
    for _sp, _y_key in [('train_df','y_train'), ('val_df','y_val'), ('test_df','y_test')]:
        _tmp = cat[_sp][[_dev_col, _date_col]].copy()
        _y = cat[_y_key]
        # SMOTE augments y_train beyond train_df length — clip to original rows only
        if len(_y) != len(cat[_sp]):
            _y = _y[:len(cat[_sp])]
        _tmp['_y']  = _y
        _tmp['_sp'] = _sp
        _tmp['_ri'] = np.arange(len(_tmp))
        _parts.append(_tmp)

    _df = pd.concat(_parts, ignore_index=True)
    # Parse date: try YYYYMMDD integer first, then generic
    _ser = _df[_date_col].astype(str).str[:8]
    _parsed = pd.to_datetime(_ser, format='%Y%m%d', errors='coerce')
    if _parsed.isna().mean() > 0.5:
        _parsed = pd.to_datetime(_df[_date_col], errors='coerce')
    _df[_date_col] = _parsed
    _df = _df.sort_values([_dev_col, _date_col]).reset_index(drop=True)

    # Shift label by 1 within each device — excludes current row from its own window
    _df['_lbl'] = _df.groupby(_dev_col)['_y'].shift(1).fillna(0).astype(np.int8)

    # Rolling failure counts
    for _w, _col in [(7,'roll_fail_7d'), (30,'roll_fail_30d'), (90,'roll_fail_90d')]:
        _df[_col] = (
            _df.groupby(_dev_col)['_lbl']
               .transform(lambda x, w=_w: x.rolling(w, min_periods=0).sum())
               .astype(np.float32)
        )

    # Days since last failure (forward-fill last-failure-date per device)
    _df['_lfd'] = _df[_date_col].where(_df['_lbl'] == 1)
    _df['_lfd'] = _df.groupby(_dev_col)['_lfd'].transform('ffill')
    _df['days_since_fail'] = (
        (_df[_date_col] - _df['_lfd']).dt.days
          .fillna(999).clip(upper=999).astype(np.float32)
    )

    # Failure-free streak (consecutive no-fail days, resets on each failure)
    def _streak(s):
        out, c = np.empty(len(s), np.float32), 0.0
        for i, v in enumerate(s.values):
            c = 0.0 if v else c + 1.0
            out[i] = c
        return pd.Series(out, index=s.index)
    _df['fail_free_streak'] = _df.groupby(_dev_col)['_lbl'].transform(_streak)

    # Split back by original tag + row index
    _out = {}
    for _sp in ['train_df', 'val_df', 'test_df']:
        _rows = (_df[_df['_sp'] == _sp]
                   .sort_values('_ri')[_ROLL_NEW_COLS].values.astype(np.float32))
        _out[_sp] = _rows
    return _out, _dev_col, _date_col


# ── Extend FEATURE_COLS once (idempotent) ────────────────────────────────────
for _c in _ROLL_NEW_COLS:
    if _c not in FEATURE_COLS:
        FEATURE_COLS.append(_c)
_expected_cols = len(FEATURE_COLS)

# ── Apply per category — skip if already extended (per-category shape guard) ─
for _cat_name in list(CATEGORIES.keys()):
    _cat = CATEGORIES[_cat_name]

    if _cat['X_train'].shape[1] == _expected_cols:
        print(f"[{_cat_name}] Already at {_expected_cols} cols — skipping.")
        continue

    try:
        _arrs, _dcol, _dtcol = _roll_feats_for_category(_cat)
        _n_orig = len(_cat['train_df'])
        _n_smote_extra = _cat['X_train'].shape[0] - _n_orig

        if _n_smote_extra > 0:
            # SMOTE synthetic rows: fill with mean of positive-class rolling features
            _pos_mask = _cat['y_train'][:_n_orig] == 1
            _pos_mean = _arrs['train_df'][_pos_mask].mean(axis=0, keepdims=True)
            _smote_fill = np.repeat(_pos_mean, _n_smote_extra, axis=0)
            _train_full = np.vstack([_arrs['train_df'], _smote_fill])
        else:
            _train_full = _arrs['train_df']

        # X_cv must match its actual row count
        _cv_roll = np.vstack([_train_full, _arrs['val_df']])

        _cat['X_train'] = np.hstack([_cat['X_train'], _train_full])
        _cat['X_val']   = np.hstack([_cat['X_val'],   _arrs['val_df']])
        _cat['X_test']  = np.hstack([_cat['X_test'],  _arrs['test_df']])
        _cat['X_cv']    = np.hstack([_cat['X_cv'],    _cv_roll])

        _n_orig_pos_mask = _cat['y_train'][:_n_orig] == 1
        print(f"[{_cat_name}] device={_dcol}  date={_dtcol}")
        print(f"  roll_fail_7d  — all:{_arrs['train_df'][:,0].mean():.3f}"
              f"  pos-class:{_arrs['train_df'][_n_orig_pos_mask,0].mean():.3f}")
        print(f"  days_since_fail  — mean: {_arrs['train_df'][:,3].mean():.1f}")
        print(f"  fail_free_streak — mean: {_arrs['train_df'][:,4].mean():.1f}")
        print(f"  X_train:{_cat['X_train'].shape}  X_val:{_cat['X_val'].shape}"
              f"  X_test:{_cat['X_test'].shape}  X_cv:{_cat['X_cv'].shape}")
    except Exception as _ex:
        import traceback
        print(f"[{_cat_name}] Rolling features FAILED: {_ex}")
        traceback.print_exc()

print(f"\nFEATURE_COLS: {len(FEATURE_COLS)} total  (rolling cols: {_ROLL_NEW_COLS})")


In [ ]:
# ── CELL 18 : MODEL 2 — Random Forest (per category) ────────────────────────────
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    with mlflow.start_run(run_name=f'lr_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'linear_baseline'})
        lr = Pipeline([('sc', StandardScaler()),
                        ('clf', LogisticRegression(C=0.1, class_weight='balanced',
                                                   max_iter=1000, solver='lbfgs', random_state=42))])
        lr.fit(X_train, y_train)
        mlflow.log_params({'C': 0.1, 'class_weight': 'balanced', 'scaler': 'standard'})
        all_m = evaluate_all(lr, f'lr_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(np.abs(lr.named_steps['clf'].coef_[0]), f'lr_{tag}')
        mlflow.sklearn.log_model(lr, 'model',
                                 signature=infer_signature(X_train, lr.predict_proba(X_train)),
                                 input_example=X_train[:3])
        CATEGORY_RESULTS[CATEGORY]['lr_run'] = run.info.run_id
    print(f'  [{CATEGORY}] LR   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 19 : MODEL 3 — XGBoost baseline (per category) ─────────────────────────
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    with mlflow.start_run(run_name=f'rf_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'bagging_ensemble'})
        rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=20,
                                     class_weight='balanced_subsample', n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        mlflow.log_params({'n_estimators': 300, 'max_depth': 12, 'class_weight': 'balanced_subsample'})
        all_m = evaluate_all(rf, f'rf_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(rf.feature_importances_, f'rf_{tag}')
        mlflow.sklearn.log_model(rf, 'model',
                                 signature=infer_signature(X_train, rf.predict_proba(X_train)),
                                 input_example=X_train[:3])
        CATEGORY_RESULTS[CATEGORY]['rf_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['rf_model']  = rf  # needed for honest stacking
    print(f'  [{CATEGORY}] RF   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 20 : MODEL 4 — LightGBM (per category) ─────────────────────────────────
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    XGB_PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=10, gamma=1.0, reg_alpha=0.1,
                      reg_lambda=1.0, scale_pos_weight=SCALE_POS_WEIGHT, eval_metric='auc',
                      early_stopping_rounds=40, tree_method='hist', random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=f'xgb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_xgb'})
        xgb = XGBClassifier(**XGB_PARAMS)
        xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
        mlflow.log_params({**XGB_PARAMS, 'best_iter': xgb.best_iteration})
        all_m = evaluate_all(xgb, f'xgb_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(xgb.feature_importances_, f'xgb_{tag}')
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS[:X_train.shape[1]]), xgb.predict_proba(X_train))
        mlflow.xgboost.log_model(xgb, 'model', signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS[:X_train.shape[1]]))
        CATEGORY_RESULTS[CATEGORY]['xgb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] XGB  AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}  best_iter={xgb.best_iteration}')


In [ ]:
# ── CELL 21 : MODEL 5 — CatBoost (per category) ─────────────────────────────────
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    LGB_PARAMS = dict(n_estimators=600, num_leaves=63, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                      scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1,
                      metric='auc')  # auc early-stopping prevents trivial all-zero convergence
    with mlflow.start_run(run_name=f'lgb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_lgbm'})
        lgb = LGBMClassifier(**LGB_PARAMS)
        lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                callbacks=[early_stopping(40, verbose=False), log_evaluation(100)])
        mlflow.log_params({**LGB_PARAMS, 'best_iter': lgb.best_iteration_})
        all_m = evaluate_all(lgb, f'lgb_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(lgb.feature_importances_, f'lgb_{tag}')
        mlflow.sklearn.log_model(lgb, 'model',
                                 signature=infer_signature(X_train, lgb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['lgb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] LGB  AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}  best_iter={lgb.best_iteration_}')

In [ ]:
# ── CELL 22 : MODEL 6 — ExtraTreesClassifier (per category) ─────────────────────
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    cat_idx = [i for i, c in enumerate(FEATURE_COLS)
               if any(k in c for k in ('device_type', 'facility', 'subsystem', 'enc'))]
    CB_PARAMS = dict(iterations=600, depth=7, learning_rate=0.05, l2_leaf_reg=3,
                     scale_pos_weight=SCALE_POS_WEIGHT, eval_metric='AUC',
                     early_stopping_rounds=40, random_seed=42, task_type='CPU', verbose=100)
    with mlflow.start_run(run_name=f'cb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_catboost'})
        cb = CatBoostClassifier(**CB_PARAMS)
        cb.fit(X_train, y_train, eval_set=(X_val, y_val),
               cat_features=cat_idx if cat_idx else None)
        mlflow.log_params({**{k: v for k, v in CB_PARAMS.items() if k != 'verbose'},
                           'best_iter': cb.best_iteration_, 'cat_features_count': len(cat_idx)})
        all_m = evaluate_all(cb, f'cb_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(cb.get_feature_importance(), f'cb_{tag}')
        cb.save_model(f'/tmp/ps1_{tag}_catboost.cbm')
        mlflow.log_artifact(f'/tmp/ps1_{tag}_catboost.cbm', 'catboost_native')
        mlflow.sklearn.log_model(cb, 'model',
                                 signature=infer_signature(X_train, cb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['cb_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['cb_model']  = cb  # needed for honest stacking
    print(f'  [{CATEGORY}] CB   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}  best_iter={cb.best_iteration_}')


In [ ]:
# ── CELL 23 : MODEL 7 — BalancedRandomForestClassifier (per category) ───────────
# Extremely randomized trees: random thresholds at each split give more variance
# reduction than standard RF. class_weight='balanced_subsample' recomputes
# sample weights per bootstrap — handles class imbalance without SMOTE.
from sklearn.ensemble import ExtraTreesClassifier

for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    ET_PARAMS = dict(n_estimators=500, max_depth=None, min_samples_leaf=5,
                     class_weight='balanced_subsample', max_features='sqrt',
                     random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=f'et_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'ensemble_extra_trees'})
        mlflow.log_params(ET_PARAMS)
        et = ExtraTreesClassifier(**ET_PARAMS)
        et.fit(X_train, y_train)
        all_m = evaluate_all(et, f'et_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(et.feature_importances_, f'et_{tag}')
        mlflow.sklearn.log_model(et, 'model',
                                 signature=infer_signature(X_train, et.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['et_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['et_model'] = et
    print(f'  [{CATEGORY}] ET   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  '
          f'Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 24 : MODEL 8 — HistGradientBoostingClassifier (per category) ───────────
# Each tree's bootstrap is class-balanced: resamples minority to match majority
# within every tree. Directly solves class imbalance without SMOTE pre-processing
# overhead — each of 500 trees sees ~50% positive examples.
from imblearn.ensemble import BalancedRandomForestClassifier

for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    BRF_PARAMS = dict(n_estimators=500, max_depth=None, min_samples_leaf=5,
                      sampling_strategy='not majority', replacement=True,
                      random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=f'brf_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'ensemble_balanced_rf'})
        mlflow.log_params(BRF_PARAMS)
        brf = BalancedRandomForestClassifier(**BRF_PARAMS)
        brf.fit(X_train, y_train)
        all_m = evaluate_all(brf, f'brf_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(brf.feature_importances_, f'brf_{tag}')
        mlflow.sklearn.log_model(brf, 'model',
                                 signature=infer_signature(X_train, brf.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['brf_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['brf_model'] = brf
    print(f'  [{CATEGORY}] BRF  AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  '
          f'Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 25 : Optuna — XGBoost + LightGBM HP tuning (per category) ──────────────
# sklearn's native GBDT: native NaN handling (no imputation needed for NaN
# features). Also used as a base learner in stacking (Cell 18); tracked here
# as a standalone model for leaderboard ranking.
from sklearn.ensemble import HistGradientBoostingClassifier

for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    HGBM_PARAMS = dict(max_iter=500, max_depth=6, learning_rate=0.05,
                       min_samples_leaf=20, l2_regularization=0.1,
                       class_weight='balanced', random_state=42)
    with mlflow.start_run(run_name=f'hgbm_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_hgbm'})
        mlflow.log_params(HGBM_PARAMS)
        hgbm = HistGradientBoostingClassifier(**HGBM_PARAMS)
        hgbm.fit(X_train, y_train)   # native NaN support; no eval_set needed
        all_m = evaluate_all(hgbm, f'hgbm_{tag}')
        mlflow.log_metrics(all_m)
        # HGBM exposes feature_importances_ only via permutation — skip log_feat_importance
        mlflow.sklearn.log_model(hgbm, 'model',
                                 signature=infer_signature(X_train, hgbm.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['hgbm_run']   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]['hgbm_model'] = hgbm
    print(f'  [{CATEGORY}] HGBM AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  '
          f'Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 26 : MLflow leaderboard — search_runs per experiment (ranked by test_ap) ───
# SPEED FIX: best params saved to checkpoints/optuna_params.joblib after first
# run. Every subsequent run loads params and skips Optuna entirely (~0 sec).
# Set FORCE_OPTUNA=True to re-run the search (e.g. after major data changes).
# Trials: XGB=50 (was 150), LGB=40 (was 100). CV splits=3 (was 5). Timeout=20min.

import os as _os, joblib as _jl_opt

_OPT_CKPT    = "/home/sagemaker-user/PS1V/checkpoints/optuna_params.joblib"
FORCE_OPTUNA = False
_RERUN_KEYS  = set()  # VALIDATOR: loads from checkpoint if present; set FORCE_OPTUNA=True to re-run
_saved_params = {}
if not FORCE_OPTUNA and _os.path.exists(_OPT_CKPT):
    _saved_params = _jl_opt.load(_OPT_CKPT)
    print(f"[OPTUNA] Loaded saved params: {list(_saved_params.keys())}")
    print("  Set FORCE_OPTUNA=True to re-run search.")

for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()
    TS_CV = TimeSeriesSplit(n_splits=3)

    # ── XGBoost Optuna ───────────────────────────────────────────────────────
    _xgb_key = f"xgb_{tag}"
    _xgb_study = None  # only set if search runs (used for plots)
    if _xgb_key in _saved_params and not FORCE_OPTUNA and _xgb_key not in _RERUN_KEYS:
        BEST_PARAMS = {**_saved_params[_xgb_key]["best_params"],
                       "scale_pos_weight": SCALE_POS_WEIGHT,
                       "tree_method": "hist", "random_state": 42, "n_jobs": -1}
        _xgb_cv_ap = _saved_params[_xgb_key].get("best_cv_ap",
                                          _saved_params[_xgb_key].get("best_ap", 0.0))
        print(f"\n  [{CATEGORY}] XGB: saved params loaded (CV_AP={_xgb_cv_ap:.4f}) — search skipped")
    else:
        print(f"\n  [{CATEGORY}] XGB: running Optuna (50 trials, timeout=20min) ...")
        def optuna_objective(trial):
            p = dict(
                n_estimators          = trial.suggest_int("n_estimators", 200, 800),
                max_depth             = trial.suggest_int("max_depth", 4, 9),
                learning_rate         = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                subsample             = trial.suggest_float("subsample", 0.5, 1.0),
                colsample_bytree      = trial.suggest_float("colsample_bytree", 0.5, 1.0),
                min_child_weight      = trial.suggest_int("min_child_weight", 1, 30),
                gamma                 = trial.suggest_float("gamma", 0.0, 3.0),
                reg_alpha             = trial.suggest_float("reg_alpha", 1e-4, 10., log=True),
                reg_lambda            = trial.suggest_float("reg_lambda", 1e-4, 10., log=True),
                scale_pos_weight      = SCALE_POS_WEIGHT,
                eval_metric           = "aucpr",
                early_stopping_rounds = 30,
                tree_method="hist", random_state=42, n_jobs=-1)
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
                m = XGBClassifier(**p)
                m.fit(X_cv[tr], y_cv[tr], eval_set=[(X_cv[va], y_cv[va])], verbose=False)
                ap = average_precision_score(y_cv[va], m.predict_proba(X_cv[va])[:, 1])
                scores.append(ap)
                trial.report(np.mean(scores), step=step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return np.mean(scores)

        _xgb_study = optuna.create_study(
            direction="maximize", study_name=f"ps1-xgb-{tag}",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
        optuna.logging.set_verbosity(optuna.logging.INFO)
        optuna.logging.set_verbosity(optuna.logging.INFO)
        _xgb_study.optimize(optuna_objective, n_trials=100, timeout=1800, show_progress_bar=False)
        _xgb_cv_ap = _xgb_study.best_value
        BEST_PARAMS = {**_xgb_study.best_params,
                       "scale_pos_weight": SCALE_POS_WEIGHT,
                       "tree_method": "hist", "random_state": 42, "n_jobs": -1}
        _saved_params[_xgb_key] = {"best_params": _xgb_study.best_params,
                                    "best_cv_ap": _xgb_cv_ap}
        _jl_opt.dump(_saved_params, _OPT_CKPT)
        print(f"  [{CATEGORY}] XGB params saved → {_OPT_CKPT}")

    CATEGORY_RESULTS[CATEGORY]["best_params"] = BEST_PARAMS

    with mlflow.start_run(run_name=f"xgb_optuna_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "xgboost_optuna_tuned"})
        mlflow.log_params({**BEST_PARAMS, "optuna_trials": 100, "optuna_cv_folds": 3,
                           "optuna_objective": "average_precision",
                           "optuna_best_cv_ap": round(_xgb_cv_ap, 4), "smote": False})
        xgb_tuned = XGBClassifier(**BEST_PARAMS)
        xgb_tuned.fit(X_cv, y_cv, verbose=False)
        all_m = evaluate_all(xgb_tuned, f"xgb_optuna_{tag}")
        mlflow.log_metrics({**all_m, "optuna_cv_ap": round(_xgb_cv_ap, 4)})
        log_feat_importance(xgb_tuned.feature_importances_, f"xgb_optuna_{tag}")
        if _xgb_study is not None:
            for fname, fig_fn in [
                (f"optuna_history_{tag}.png",      optuna.visualization.matplotlib.plot_optimization_history),
                (f"optuna_param_import_{tag}.png", optuna.visualization.matplotlib.plot_param_importances)]:
                try:
                    fig_fn(_xgb_study); plt.tight_layout()
                    plt.savefig(f"/tmp/{fname}", dpi=80); plt.close()
                    mlflow.log_artifact(f"/tmp/{fname}")
                except Exception: pass
        _fc = FEATURE_COLS[:X_train.shape[1]]  # guard: checkpoint may have fewer cols
        sig = infer_signature(pd.DataFrame(X_train, columns=_fc),
                              xgb_tuned.predict_proba(X_train))
        try:
            mlflow.xgboost.log_model(xgb_tuned, "model", signature=sig,
                                     input_example=pd.DataFrame(X_train[:3], columns=_fc))
        except Exception as _lme: print(f"  [warn] log_model: {_lme}")
        CATEGORY_RESULTS[CATEGORY]["xgb_tuned_run"]   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]["xgb_tuned_model"] = xgb_tuned
    print(f"  [{CATEGORY}] XGB  CV_AP={_xgb_cv_ap:.4f}  AUC={all_m['test_auc']:.4f}  AP={all_m['test_ap']:.4f}  Rec={all_m['test_rec']:.4f}")

    # ── LightGBM Optuna ──────────────────────────────────────────────────────
    _lgb_key = f"lgb_{tag}"
    _lgb_study = None
    if _lgb_key in _saved_params and not FORCE_OPTUNA and _lgb_key not in _RERUN_KEYS:
        BEST_LGB_PARAMS = {**_saved_params[_lgb_key]["best_params"],
                           "scale_pos_weight": SCALE_POS_WEIGHT,
                           "metric": "binary_logloss",
                           "n_jobs": -1, "random_state": 42, "verbose": -1}
        _lgb_cv_ap = _saved_params[_lgb_key].get("best_cv_ap",
                                          _saved_params[_lgb_key].get("best_ap", 0.0))
        print(f"\n  [{CATEGORY}] LGB: saved params loaded (CV_AP={_lgb_cv_ap:.4f}) — search skipped")
    else:
        print(f"\n  [{CATEGORY}] LGB: running Optuna (40 trials, timeout=20min) ...")
        def lgb_objective(trial):
            p = dict(
                n_estimators      = trial.suggest_int("n_estimators", 200, 800),
                max_depth         = trial.suggest_int("max_depth", 4, 10),
                learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                num_leaves        = trial.suggest_int("num_leaves", 20, 200),
                subsample         = trial.suggest_float("subsample", 0.5, 1.0),
                colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
                min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
                reg_alpha         = trial.suggest_float("reg_alpha", 1e-4, 10., log=True),
                reg_lambda        = trial.suggest_float("reg_lambda", 1e-4, 10., log=True),
                scale_pos_weight  = SCALE_POS_WEIGHT,
                n_jobs=-1, random_state=42, verbose=-1)  # no metric: n_estimators in search; early stopping removed
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
                m = LGBMClassifier(**p)
                m.fit(X_cv[tr], y_cv[tr])  # n_estimators tuned by Optuna; no early stopping
                ap = average_precision_score(y_cv[va], m.predict_proba(X_cv[va])[:, 1])
                scores.append(ap)
                trial.report(np.mean(scores), step=step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return np.mean(scores)

        _lgb_study = optuna.create_study(
            direction="maximize", study_name=f"ps1-lgb-{tag}",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
        optuna.logging.set_verbosity(optuna.logging.INFO)
        _lgb_study.optimize(lgb_objective, n_trials=60, timeout=1200, show_progress_bar=False)
        _lgb_cv_ap = _lgb_study.best_value
        BEST_LGB_PARAMS = {**_lgb_study.best_params,
                           "scale_pos_weight": SCALE_POS_WEIGHT,
                           "metric": "binary_logloss",
                           "n_jobs": -1, "random_state": 42, "verbose": -1}
        _saved_params[_lgb_key] = {"best_params": _lgb_study.best_params,
                                    "best_cv_ap": _lgb_cv_ap}
        _jl_opt.dump(_saved_params, _OPT_CKPT)
        print(f"  [{CATEGORY}] LGB params saved → {_OPT_CKPT}")

    CATEGORY_RESULTS[CATEGORY]["best_lgb_params"] = BEST_LGB_PARAMS

    with mlflow.start_run(run_name=f"lgb_optuna_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "lgbm_optuna_tuned"})
        mlflow.log_params({**BEST_LGB_PARAMS, "optuna_trials": 60, "optuna_cv_folds": 3,
                           "optuna_objective": "average_precision",
                           "optuna_best_cv_ap": round(_lgb_cv_ap, 4)})
        lgb_tuned = LGBMClassifier(**BEST_LGB_PARAMS)
        lgb_tuned.fit(X_cv, y_cv)
        all_m = evaluate_all(lgb_tuned, f"lgb_optuna_{tag}")
        mlflow.log_metrics({**all_m, "optuna_cv_ap": round(_lgb_cv_ap, 4)})
        log_feat_importance(lgb_tuned.feature_importances_, f"lgb_optuna_{tag}")
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS[:X_train.shape[1]]),
                              lgb_tuned.predict_proba(X_train))
        try:
            mlflow.sklearn.log_model(lgb_tuned, "model", signature=sig,
                                     input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS[:X_train.shape[1]]),
                                     skops_trusted_types=["collections.OrderedDict",
                                                          "lightgbm.basic.Booster",
                                                          "lightgbm.sklearn.LGBMClassifier"])
        except Exception as _lme: print(f"  [warn] log_model: {_lme}")
        CATEGORY_RESULTS[CATEGORY]["lgb_tuned_run"]   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]["lgb_tuned_model"] = lgb_tuned
    print(f"  [{CATEGORY}] LGB  CV_AP={_lgb_cv_ap:.4f}  AUC={all_m['test_auc']:.4f}  AP={all_m['test_ap']:.4f}  Rec={all_m['test_rec']:.4f}")

print(f"\n[OPTUNA] Complete. Params at {_OPT_CKPT}")
print("  Next restart: params load automatically — Optuna skipped for all 4 keys.")
print("  Set FORCE_OPTUNA=True to re-run search.")


In [ ]:
# ── CELL 27 : Honest Stacking + Isotonic Calibration (per category) ─────────────
for CATEGORY in ['VALIDATOR']:
    cat = CATEGORIES[CATEGORY]
    mlflow.set_experiment(cat['experiment_name'])
    runs = mlflow.search_runs(
        experiment_names=[cat['experiment_name']],
        filter_string="tags.model_type != 'shap_analysis' AND tags.model_type != 'xgboost_multiclass'",
        order_by=['metrics.test_ap DESC'])
    print(f'\n── {CATEGORY} base model leaderboard (ranked by test_ap) ──')
    # Deduplicate: keep latest run per model name (handles notebook re-runs).
    if 'tags.mlflow.runName' in runs.columns and 'start_time' in runs.columns:
        runs = (runs.sort_values('start_time', ascending=False)
                    .drop_duplicates('tags.mlflow.runName')
                    .sort_values('metrics.test_ap', ascending=False, ignore_index=True))
    cols = ['tags.mlflow.runName', 'metrics.test_auc', 'metrics.test_ap',
            'metrics.test_f1', 'metrics.test_prec', 'metrics.test_rec',
            'metrics.train_auc', 'metrics.val_auc']
    available = [c for c in cols if c in runs.columns]
    print(runs[available].head(10).to_string(index=False))

In [ ]:
# ── CELL 28 : SHAP — feature attribution (per category) ─────────────────────────
# Replaces sklearn StackingClassifier OOF approach.
# Root cause of prior MetaLR collapse (AUC=0.36):
#   1. class_weight="balanced" on meta-LR caused signal inversion.
#   2. OOF distribution mismatch: early-fold models produce different proba scales.
#   3. eval_set contamination: models overfit to X_val via early stopping.
#
# Fix: retrain base models with fixed n_estimators (no eval_set) on X_train →
#      honest X_val predictions for meta-LR.  Same on X_cv for X_test predictions.
# For GATE: apply SMOTE to (pre-SMOTE train + val) combined before full-data retrain.
# Calibration: isotonic CalibratedClassifierCV(lgb_tuned, cv="prefit") on (X_val, y_val).

import warnings
from sklearn.calibration import CalibratedClassifierCV, calibration_curve as _cal_curve
from sklearn.preprocessing import StandardScaler

try:
    mlflow.end_run()
except Exception:
    pass

_L0 = ["XGB", "CatBoost", "RF", "LGB"]

for CATEGORY in ["VALIDATOR"]:
    cat             = CATEGORIES[CATEGORY]
    X_train         = cat["X_train"];  y_train = cat["y_train"]
    X_val           = cat["X_val"];    y_val   = cat["y_val"]
    X_test          = cat["X_test"];   y_test  = cat["y_test"]
    X_cv            = cat["X_cv"];     y_cv    = cat["y_cv"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    print(f"\n{'='*64}")
    print(f"[{CATEGORY}] Honest Stacking + Calibration")
    print(f"{'='*64}")

    # ── Get trained model objects from CATEGORY_RESULTS ─────────────────────
    xgb_champ = CATEGORY_RESULTS[CATEGORY]["xgb_tuned_model"]
    lgb_champ = CATEGORY_RESULTS[CATEGORY]["lgb_tuned_model"]
    rf_champ  = CATEGORY_RESULTS[CATEGORY].get("rf_model")
    cb_champ  = CATEGORY_RESULTS[CATEGORY].get("cb_model")
    if rf_champ is None:
        rf_champ = mlflow.sklearn.load_model(f"runs:/{CATEGORY_RESULTS[CATEGORY]['rf_run']}/model")
        CATEGORY_RESULTS[CATEGORY]["rf_model"] = rf_champ
    if cb_champ is None:
        cb_champ = mlflow.sklearn.load_model(f"runs:/{CATEGORY_RESULTS[CATEGORY]['cb_run']}/model")
        CATEGORY_RESULTS[CATEGORY]["cb_model"] = cb_champ

    # ── Extract best iterations (guard against 0 when no early stopping) ────
    _xgb_n = int(getattr(xgb_champ, "best_ntree_limit", 0) or
                 (getattr(xgb_champ, "best_iteration", 0) + 1) or 0)
    if _xgb_n <= 0:
        try:    _xgb_n = xgb_champ.get_booster().num_boosted_rounds()
        except: _xgb_n = 300
    _xgb_n = max(_xgb_n, 1)

    _cb_n = int(getattr(cb_champ, "best_iteration_", 0) or 0)
    if _cb_n <= 0:
        try:    _cb_n = int(cb_champ.tree_count_)
        except: _cb_n = 300
    _cb_n = max(_cb_n, 1)

    _lgb_n = int(getattr(lgb_champ, "best_iteration_", 0) or 0)
    if _lgb_n <= 0:
        try:    _lgb_n = lgb_champ.booster_.num_trees()
        except: _lgb_n = 300
    _lgb_n = max(_lgb_n, 1)

    print(f"  Honest n_estimators: XGB={_xgb_n}  CB={_cb_n}  LGB={_lgb_n}  RF=clean")

    # ── Build honest param dicts (no eval_set, no early_stopping_rounds) ────
    _lgb_best = {**CATEGORY_RESULTS[CATEGORY].get("best_lgb_params", {}),
                 "objective": "binary", "n_estimators": _lgb_n,
                 "scale_pos_weight": SCALE_POS_WEIGHT,
                 "random_state": 42, "n_jobs": -1, "verbose": -1}
    for _k in ["metric", "early_stopping_rounds"]:
        _lgb_best.pop(_k, None)

    _xgb_best = {"n_estimators": _xgb_n, "learning_rate": 0.05, "max_depth": 6,
                 "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10,
                 "scale_pos_weight": SCALE_POS_WEIGHT, "tree_method": "hist",
                 "random_state": 42, "n_jobs": -1, "verbosity": 0}
    _cb_best  = {"iterations": _cb_n, "learning_rate": 0.05, "depth": 7,
                 "scale_pos_weight": SCALE_POS_WEIGHT, "random_seed": 42, "verbose": 0}

    # ── Retrain on X_train (no eval_set) → honest X_val predictions ─────────
    _xgb_tr = XGBClassifier(**_xgb_best);         _xgb_tr.fit(X_train, y_train)
    _cb_tr  = CatBoostClassifier(**_cb_best);       _cb_tr.fit(X_train, y_train)
    _lgb_tr = LGBMClassifier(**_lgb_best);          _lgb_tr.fit(X_train, y_train)

    _va_stack = np.column_stack([
        _xgb_tr.predict_proba(X_val)[:, 1],
        _cb_tr.predict_proba(X_val)[:, 1],
        rf_champ.predict_proba(X_val)[:, 1],   # RF: no early stopping, already clean
        _lgb_tr.predict_proba(X_val)[:, 1],
    ])

    # Contamination check (gap <= 0.03 = clean)
    _va_aucs = [roc_auc_score(y_val, _va_stack[:, i]) for i in range(4)]
    _te_ref  = [roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
                for m in [xgb_champ, cb_champ, rf_champ, lgb_champ]]
    print("\nContamination check (val AUC vs test AUC, gap <= 0.03 = clean):")
    for _n, _va, _te in zip(_L0, _va_aucs, _te_ref):
        _g = _va - _te
        print(f"  {_n:<12}: val={_va:.4f}  test={_te:.4f}  gap={_g:+.4f}"
              + ("  OK" if _g <= 0.03 else "  WARN (still contaminated)"))

    # ── Full-data test predictions (SMOTE of train+val for GATE) ────────────
    if CATEGORY == "GATE" and "X_train_pre_smote" in CATEGORY_RESULTS[CATEGORY]:
        _X_pre = CATEGORY_RESULTS[CATEGORY]["X_train_pre_smote"]
        _y_pre = CATEGORY_RESULTS[CATEGORY]["y_train_pre_smote"]
        from imblearn.over_sampling import SMOTE as _SMOTE_H
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _X_full, _y_full = _SMOTE_H(
                sampling_strategy=SMOTE_RATIO, random_state=42
            ).fit_resample(np.vstack([_X_pre, X_val]),
                           np.concatenate([_y_pre, y_val]))
        _spw_full = (1 - _y_full.mean()) / _y_full.mean()
        print(f"\n  GATE full-data SMOTE: {len(_X_full):,} rows  spw={_spw_full:.1f}")
    else:
        _X_full, _y_full = X_cv, y_cv
        _spw_full = SCALE_POS_WEIGHT

    _xgb_tf = XGBClassifier(**{**_xgb_best, "scale_pos_weight": _spw_full})
    _xgb_tf.fit(_X_full, _y_full)
    _cb_tf  = CatBoostClassifier(**{**_cb_best, "scale_pos_weight": _spw_full})
    _cb_tf.fit(_X_full, _y_full)
    _lgb_tf = LGBMClassifier(**{**_lgb_best, "scale_pos_weight": _spw_full})
    _lgb_tf.fit(_X_full, _y_full)
    _rf_tf  = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=20,
                                      class_weight="balanced_subsample", n_jobs=-1, random_state=42)
    _rf_tf.fit(_X_full, _y_full)

    _te_stack = np.column_stack([
        _xgb_tf.predict_proba(X_test)[:, 1],
        _cb_tf.predict_proba(X_test)[:, 1],
        _rf_tf.predict_proba(X_test)[:, 1],
        _lgb_tf.predict_proba(X_test)[:, 1],
    ])

    # ── Ensemble A: Simple average ───────────────────────────────────────────
    proba_avg = _te_stack.mean(axis=1)

    # ── Ensemble B: Val-AUC weighted average ─────────────────────────────────
    _w = np.clip(np.array(_va_aucs) - np.array(_va_aucs).min(), 0, None)
    _w = _w / (_w.sum() + 1e-9)
    proba_wavg = (_te_stack * _w).sum(axis=1)
    print("\nVal-AUC weights (honest): " +
          "  ".join(f"{n}={w:.3f}" for n, w in zip(_L0, _w)))

    # ── Ensemble C: Meta-LR (no class_weight — SMOTE already balanced) ───────
    _sc     = StandardScaler()
    _va_sc  = _sc.fit_transform(_va_stack)
    _te_sc  = _sc.transform(_te_stack)
    meta_lr = LogisticRegression(C=0.1, max_iter=2000, random_state=42, n_jobs=-1)
    meta_lr.fit(_va_sc, y_val)
    proba_stk = meta_lr.predict_proba(_te_sc)[:, 1]

    print("\nMeta-LR coefficients (should all be positive — negative = contamination):")
    for _n, _c in zip(_L0, meta_lr.coef_[0]):
        print(f"  {_n:<12}: {_c:+.4f}  {'OK' if _c > 0 else 'WARN'}")

    # ── Ensemble D: Meta-LGB ─────────────────────────────────────────────────
    meta_lgb = LGBMClassifier(n_estimators=200, num_leaves=8, max_depth=3,
                               scale_pos_weight=SCALE_POS_WEIGHT,
                               reg_lambda=5.0, min_child_samples=20,
                               random_state=42, n_jobs=-1, verbose=-1)
    meta_lgb.fit(_va_sc, y_val)
    proba_meta_lgb = meta_lgb.predict_proba(_te_sc)[:, 1]

    # ── Auto-selector ────────────────────────────────────────────────────────
    _single_auc = roc_auc_score(y_test, lgb_champ.predict_proba(X_test)[:, 1])
    _single_ap  = average_precision_score(y_test, lgb_champ.predict_proba(X_test)[:, 1])
    _ens_aucs = {
        "A-Avg"    : roc_auc_score(y_test, proba_avg),
        "B-WtdAvg" : roc_auc_score(y_test, proba_wavg),
        "C-MetaLR" : roc_auc_score(y_test, proba_stk),
        "D-MetaLGB": roc_auc_score(y_test, proba_meta_lgb),
    }
    _ens_aps = {
        "A-Avg"    : average_precision_score(y_test, proba_avg),
        "B-WtdAvg" : average_precision_score(y_test, proba_wavg),
        "C-MetaLR" : average_precision_score(y_test, proba_stk),
        "D-MetaLGB": average_precision_score(y_test, proba_meta_lgb),
    }
    _best_ens = max(_ens_aucs, key=_ens_aucs.get)
    _use_ens  = _ens_aucs[_best_ens] > _single_auc + CHAMPION_AP_TOL

    print(f"\nEnsemble vs LGB-Optuna (single)  AUC={_single_auc:.4f}  AP={_single_ap:.4f}")
    for _k in ["A-Avg", "B-WtdAvg", "C-MetaLR", "D-MetaLGB"]:
        print(f"  {_k:<14}: AUC={_ens_aucs[_k]:.4f}  AP={_ens_aps[_k]:.4f}  "
              f"delta={_ens_aucs[_k]-_single_auc:+.4f}")
    print(f"\n[Auto-select] Best ensemble: {_best_ens}  delta={_ens_aucs[_best_ens]-_single_auc:+.4f}")
    print(f"  -> {'Ensemble adds value. Use for ranked dispatch.'if _use_ens else 'Boosters correlated — LGB-Optuna is champion.'}")

    # ── Calibration: isotonic regression on val set ────────────────────────
    # CalibratedClassifierCV(cv="prefit") was removed in sklearn >= 1.4.
    # Use IsotonicRegression directly on val-set raw probabilities.
    from sklearn.isotonic import IsotonicRegression as _IR
    _iso_cal  = _IR(out_of_bounds="clip")
    _val_raw  = lgb_champ.predict_proba(X_val)[:, 1]
    _iso_cal.fit(_val_raw, y_val)

    class _IsotonicWrapper:
        """Wraps base LGB + fitted isotonic calibrator for predict_proba interface."""
        _estimator_type = "classifier"
        def __init__(self, base, iso):
            self.base_model = base; self.calibrator = iso
        def predict_proba(self, X):
            raw = self.base_model.predict_proba(X)[:, 1]
            cal = np.clip(self.calibrator.transform(raw), 0, 1)
            return np.column_stack([1 - cal, cal])
        def predict(self, X):
            return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    _calib_model = _IsotonicWrapper(lgb_champ, _iso_cal)
    proba_calib  = _calib_model.predict_proba(X_test)[:, 1]

    _raw_prob   = lgb_champ.predict_proba(X_test)[:, 1]
    _brier_raw  = float(np.mean((_raw_prob - y_test)**2))
    _brier_cal  = float(np.mean((proba_calib - y_test)**2))
    print(f"\n  Calibration (isotonic, val-set prefit):")
    print(f"    Brier raw={_brier_raw:.4f}  calibrated={_brier_cal:.4f}  "
          f"({'improved' if _brier_cal < _brier_raw else 'no improvement'})")
    print(f"    Proba mean: raw={_raw_prob.mean():.4f}  cal={proba_calib.mean():.4f}"
          f"  (positive rate={y_test.mean():.4f})")
    print(f"    AUC: raw={roc_auc_score(y_test, _raw_prob):.4f}"
          f"  cal={roc_auc_score(y_test, proba_calib):.4f}")

    # ── Store all results ─────────────────────────────────────────────────────
    CATEGORY_RESULTS[CATEGORY].update({
        "va_stack"      : _va_stack, "te_stack"      : _te_stack,
        "proba_avg"     : proba_avg, "proba_wavg"    : proba_wavg,
        "proba_stk"     : proba_stk, "proba_meta_lgb": proba_meta_lgb,
        "meta_lr"       : meta_lr,   "meta_lgb"      : meta_lgb,
        "stk_scaler"    : _sc,
        "calib_model"   : _calib_model, "proba_calib": proba_calib,
        "brier_raw"     : _brier_raw,   "brier_calib": _brier_cal,
        "use_ensemble"  : _use_ens,     "best_ens"   : _best_ens,
    })

    # ── MLflow: log stacking + calibration run ───────────────────────────────
    with mlflow.start_run(run_name=f"honest_stack_calib_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "honest_stacking_calibrated"})
        mlflow.log_params({
            "l0_models"        : "XGB_tuned+CatBoost+RF+LGB_tuned",
            "meta_model"       : "LogisticRegression(C=0.1,no_class_weight)",
            "stacking_protocol": "honest_val_set",
            "calibration"      : "isotonic_prefit_val",
            "xgb_n"            : _xgb_n, "cb_n": _cb_n, "lgb_n": _lgb_n,
            "use_ensemble"     : _use_ens, "best_ens": _best_ens,
        })
        _val_auc_meta = roc_auc_score(y_val, meta_lr.predict_proba(_va_sc)[:, 1])
        _stk_m = {
            "val_auc_metalr" : _val_auc_meta,
            "test_auc"       : roc_auc_score(y_test, proba_stk),
            "test_ap"        : average_precision_score(y_test, proba_stk),
            "test_auc_avg"   : _ens_aucs["A-Avg"],
            "test_ap_avg"    : _ens_aps["A-Avg"],
            "test_auc_calib" : roc_auc_score(y_test, proba_calib),
            "test_ap_calib"  : average_precision_score(y_test, proba_calib),
            "brier_raw"      : _brier_raw, "brier_calib": _brier_cal,
            "train_auc"      : roc_auc_score(y_train, lgb_champ.predict_proba(X_train)[:, 1]),
            "val_auc"        : roc_auc_score(y_val,   lgb_champ.predict_proba(X_val)[:, 1]),
        }
        mlflow.log_metrics(_stk_m)

        # Log base LGB + isotonic calibrator as separate artifacts
        # (_IsotonicWrapper is not a standard sklearn estimator; log components)
        from mlflow.models import infer_signature
        import joblib as _jbl_cal, tempfile as _tmp, os as _os_cal
        _tmp_dir = _tmp.mkdtemp()
        try:
            mlflow.sklearn.log_model(
                lgb_champ, "calibrated_lgb",
                signature=infer_signature(X_val, lgb_champ.predict_proba(X_val)),
                input_example=X_val[:3])
            _iso_path = _os_cal.path.join(_tmp_dir, "isotonic_calibrator.joblib")
            _jbl_cal.dump(_iso_cal, _iso_path)
            mlflow.log_artifact(_iso_path, "calibration")
            mlflow.set_tag("calibration_method", "isotonic_prefit_val")
        except Exception as _ce:
            print(f"  [Warn] calibrated model log: {_ce}")

        # Log calibration curve plot
        try:
            _fig_c, _axes_c = plt.subplots(1, 2, figsize=(11, 4))
            for _ax, _p, _lbl, _col in [
                (_axes_c[0], _raw_prob,   "Uncalibrated", "orange"),
                (_axes_c[1], proba_calib, "Calibrated",   "purple"),
            ]:
                try:
                    _fp, _mp = _cal_curve(y_test, _p, n_bins=10, strategy="quantile")
                    _ax.plot(_mp, _fp, "o-", color=_col, label=_lbl)
                    _ax.plot([0,1],[0,1],"k--", alpha=0.4); _ax.legend()
                    _ax.set_title(f"{CATEGORY} — {_lbl}"); _ax.grid(True, alpha=0.3)
                except Exception: pass
            plt.tight_layout()
            mlflow.log_figure(_fig_c, "calibration_curve.png")
            plt.close(_fig_c)
        except Exception as _e:
            print(f"  [Warn] calibration plot failed: {_e}")

        CATEGORY_RESULTS[CATEGORY]["stack_run"] = run.info.run_id
        print(f"\n  MLflow stacking run: {run.info.run_id}")
        print(f"  Logged: test_ap={_stk_m['test_ap']:.4f}  "
              f"test_auc_calib={_stk_m['test_auc_calib']:.4f}  "
              f"brier_calib={_brier_cal:.4f}")

In [ ]:
# ── CELL 29 : Stage 2 — severity multiclass (per category, guarded) ─────────────
# Runs for both TVM and GATE.
# Plots (per category): bar, beeswarm, scatter (top feature), waterfall (top failure)
# Printed analysis: top-20 importance table, waterfall drivers, missed-failure SHAP,
#                   TVM vs GATE comparison table.
_shap_imp_all = {}   # {CATEGORY: pd.Series(mean_abs_shap, index=features)}

for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT    = cat["scale_pos_weight"]
    EXPERIMENT_NAME     = cat["experiment_name"]
    MLFLOW_MODEL_NAME   = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR        = cat["recall_floor"]
    COMMON_TAGS = {**make_common_tags(CATEGORY),
                   "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    xgb_tuned = CATEGORY_RESULTS[CATEGORY]['xgb_tuned_model']
    _n_shap   = min(2000, len(X_test))
    X_shap    = pd.DataFrame(X_test[:_n_shap], columns=FEATURE_COLS)
    y_shap    = np.array(y_test[:_n_shap]).astype(int)

    # XGBoost 2.x stores base_score as '[value]' -- shap.TreeExplainer cannot parse it
    # (both Booster and XGBClassifier paths call save_config() internally).
    # Use XGBoost native pred_contribs=True: identical SHAP values, no SHAP loader.
    import xgboost as _xgb19
    _booster19  = xgb_tuned.get_booster()
    _dmat19     = _xgb19.DMatrix(X_shap.values, feature_names=list(X_shap.columns))
    _contribs19 = _booster19.predict(_dmat19, pred_contribs=True)
    # pred_contribs: (n_samples, n_features + 1) -- last col = bias term
    shap_values = shap.Explanation(
        values        = _contribs19[:, :-1],
        base_values   = _contribs19[:, -1],
        data          = X_shap.values,
        feature_names = list(X_shap.columns),
    )

    # Predicted probabilities (for failure-case selection and FN/TP split)
    _probs19 = xgb_tuned.predict_proba(X_shap)[:, 1]
    _pred19  = (_probs19 >= 0.5).astype(int)

    # Mean |SHAP| importance
    _mean_abs = pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS)
    _shap_imp_all[CATEGORY] = _mean_abs.sort_values(ascending=False)
    top_feat = _shap_imp_all[CATEGORY].index[0]

    # Index of highest-confidence true positive (for waterfall)
    _tp_mask = (y_shap == 1) & (_pred19 == 1)
    if _tp_mask.any():
        _tp_cands = np.where(_tp_mask)[0]
        _tp_idx   = int(_tp_cands[np.argmax(_probs19[_tp_cands])])
    else:
        _tp_idx = int(np.argmax(_probs19))

    with mlflow.start_run(run_name=f'xgb_shap_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'shap_analysis',
                         'parent_run': CATEGORY_RESULTS[CATEGORY]['xgb_tuned_run']})
        # Bar + beeswarm (top 20 features)
        for sname, sfn in [('bar', shap.plots.bar), ('beeswarm', shap.plots.beeswarm)]:
            sfn(shap_values, max_display=20, show=False)
            fig = plt.gcf(); fig.tight_layout()
            fpath = f'/tmp/shap_{sname}_{tag}.png'
            fig.savefig(fpath, dpi=100); plt.close(fig)
            mlflow.log_artifact(fpath)
        # Scatter / dependency plot for top feature
        shap.plots.scatter(shap_values[:, top_feat], color=shap_values, show=False)
        fig = plt.gcf(); fig.tight_layout()
        fpath = f'/tmp/shap_dep_{tag}.png'
        fig.savefig(fpath, dpi=100); plt.close(fig)
        mlflow.log_artifact(fpath)
        # Waterfall for highest-confidence caught failure
        shap.plots.waterfall(shap_values[_tp_idx], max_display=15, show=False)
        fig = plt.gcf(); fig.tight_layout()
        fpath = f'/tmp/shap_waterfall_{tag}.png'
        fig.savefig(fpath, dpi=100); plt.close(fig)
        mlflow.log_artifact(fpath)
        # Importance CSV
        _shap_df = (_shap_imp_all[CATEGORY].reset_index()
                        .rename(columns={"index": "feature", 0: "mean_abs_shap"}))
        _shap_df.to_csv(f'/tmp/shap_importance_{tag}.csv', index=False)
        mlflow.log_artifact(f'/tmp/shap_importance_{tag}.csv')
        mlflow.log_param('top_shap_feature', top_feat)

    _SEP19 = "=" * 65
    # ── 1. Top-20 SHAP importance table ──────────────────────────────────────
    print(f"\n{_SEP19}")
    print(f"SHAP Feature Importance -- {CATEGORY}  (test n={_n_shap})")
    print(f"  mean |SHAP|: avg absolute impact on model output across all test samples")
    print(_SEP19)
    _top20 = _shap_imp_all[CATEGORY].head(20).reset_index()
    _top20.columns = ["feature", "mean_abs_shap"]
    _top20.insert(0, "rank", range(1, len(_top20) + 1))
    _top20["pct_total"] = (_top20["mean_abs_shap"] / _top20["mean_abs_shap"].sum() * 100).round(1)
    print(_top20.round(5).to_string(index=False))
    print(f"\n  Top feature : {top_feat}  (mean|SHAP|={_shap_imp_all[CATEGORY].iloc[0]:.5f})")

    # ── 2. Waterfall drivers (printed, top 10 for the sample failure) ────────
    _wf_sv  = pd.Series(shap_values.values[_tp_idx], index=FEATURE_COLS)
    _wf_top = _wf_sv.abs().sort_values(ascending=False).head(10)
    print(f"\n  Waterfall sample [{CATEGORY}] -- idx={_tp_idx}  "
          f"pred_prob={_probs19[_tp_idx]:.3f}  true_label={y_shap[_tp_idx]}")
    print(f"  {'Feature':<40}  {'SHAP':>9}  {'Raw value':>10}")
    print(f"  {'-'*40}  {'-'*9}  {'-'*10}")
    for feat in _wf_top.index:
        sv    = _wf_sv[feat]
        raw_v = X_shap.iloc[_tp_idx][feat]
        arrow = "+" if sv > 0 else "-"
        print(f"  {arrow} {feat:<39}  {sv:+9.5f}  {raw_v:10.3g}")

    # ── 3. Missed-failure analysis ────────────────────────────────────────────
    _fn_mask  = (y_shap == 1) & (_pred19 == 0)   # false negatives: missed failures
    _tp_mask2 = (y_shap == 1) & (_pred19 == 1)   # true positives:  caught failures
    n_tp = int(_tp_mask2.sum()); n_fn = int(_fn_mask.sum())
    print(f"\n{_SEP19}")
    print(f"Missed-Failure SHAP Analysis -- {CATEGORY}")
    print(f"  Caught (TP): {n_tp}  |  Missed (FN): {n_fn}")
    print(f"  Negative SHAP in FN = features pulling the score DOWN on missed alarms.")
    print(f"  Suppression = feature carries less SHAP weight in missed vs caught failures.")
    print(_SEP19)
    if n_fn >= 5 and n_tp >= 5:
        _sv_tp  = shap_values.values[_tp_mask2]   # (n_tp, n_feat)
        _sv_fn  = shap_values.values[_fn_mask]    # (n_fn, n_feat)
        _cmp19  = pd.DataFrame({
            "signed_TP":    _sv_tp.mean(0),
            "signed_FN":    _sv_fn.mean(0),
            "abs_TP":       np.abs(_sv_tp).mean(0),
            "abs_FN":       np.abs(_sv_fn).mean(0),
        }, index=FEATURE_COLS)
        _cmp19["suppression"] = _cmp19["abs_FN"] - _cmp19["abs_TP"]
        # Most suppressed: features that matter LESS for missed failures
        print("\n  Top 10 features suppressed in missed failures (carry less |SHAP| vs caught):")
        _supp = (_cmp19.sort_values("suppression").head(10)
                       [["abs_TP", "abs_FN", "signed_TP", "signed_FN", "suppression"]])
        print(_supp.round(5).to_string())
        # Features actively pushing score DOWN (most negative mean SHAP) in FN
        _neg_fn = _cmp19["signed_FN"].sort_values().head(10)
        print(f"\n  Top 10 features pushing score DOWN in missed failures (negative SHAP):")
        print(f"  {'Feature':<40}  {'FN SHAP':>9}  {'TP SHAP':>9}")
        print(f"  {'-'*40}  {'-'*9}  {'-'*9}")
        for feat, fn_v in _neg_fn.items():
            tp_v = _cmp19.loc[feat, "signed_TP"]
            print(f"  {feat:<40}  {fn_v:+9.5f}  {tp_v:+9.5f}")
    else:
        print(f"  [SKIP] Too few samples (FN={n_fn}, TP={n_tp}) -- need >= 5 each")

# ── 4. TVM vs GATE comparison table ──────────────────────────────────────────
if len(_shap_imp_all) == 2:
    _SEP19b = "=" * 70
    print(f"\n{_SEP19b}")
    print("TVM vs GATE -- SHAP Feature Importance Comparison")
    print("  Union of top-15 features from each model, sorted by TVM rank.")
    print("  rank_delta = |TVM_rank - GATE_rank|: large = important to one but not the other.")
    print(_SEP19b)
    _tvm_s  = _shap_imp_all["TVM"].rename("TVM")
    _gate_s = _shap_imp_all["GATE"].rename("GATE")
    _cmp2   = pd.concat([_tvm_s, _gate_s], axis=1).fillna(0.0)
    _cmp2["TVM_rank"]   = _cmp2["TVM"].rank(ascending=False, method="min").astype(int)
    _cmp2["GATE_rank"]  = _cmp2["GATE"].rank(ascending=False, method="min").astype(int)
    _cmp2["rank_delta"] = (_cmp2["TVM_rank"] - _cmp2["GATE_rank"]).abs()
    _union15 = (set(_shap_imp_all["TVM"].head(15).index)
              | set(_shap_imp_all["GATE"].head(15).index))
    _show2 = (_cmp2.loc[list(_union15)]
                   .sort_values("TVM_rank")
                   [["TVM_rank", "TVM", "GATE_rank", "GATE", "rank_delta"]])
    _show2.columns = ["TVM_rank", "TVM_|SHAP|", "GATE_rank", "GATE_|SHAP|", "rank_delta"]
    print(_show2.round(5).to_string())

    _tvm_top5  = list(_shap_imp_all["TVM"].head(5).index)
    _gate_top5 = list(_shap_imp_all["GATE"].head(5).index)
    _shared10  = sorted(set(_shap_imp_all["TVM"].head(10).index)
                      & set(_shap_imp_all["GATE"].head(10).index))
    _tvm_only  = [f for f in _tvm_top5  if f not in _shap_imp_all["GATE"].head(10).index]
    _gate_only = [f for f in _gate_top5 if f not in _shap_imp_all["TVM"].head(10).index]
    print(f"\n  TVM  top-5 : {_tvm_top5}")
    print(f"  GATE top-5 : {_gate_top5}")
    print(f"  Shared top-10 : {_shared10}")
    if _tvm_only:  print(f"  TVM-only  (top-5, not in GATE top-10) : {_tvm_only}")
    if _gate_only: print(f"  GATE-only (top-5, not in TVM  top-10) : {_gate_only}")


In [ ]:
# ── CELL 30 : F2 + Budget Threshold + Quality Gate (per category) ───────────────
# Skipped for GATE: failure_levels 2/3/4 have ~6/7/6 rows — min class < 20.
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    if 'failure_level' not in train_df.columns:
        print(f'  [{CATEGORY}] failure_level column absent — skip Stage 2'); continue

    sev_tr = train_df[train_df['will_fail_3d'] == 1].dropna(subset=['failure_level'])
    sev_te = test_df[test_df['will_fail_3d'] == 1].dropna(subset=['failure_level'])
    _class_counts = sev_tr['failure_level'].value_counts() if len(sev_tr) > 0 else pd.Series([])
    _skip = len(sev_tr) == 0 or _class_counts.min() < 20
    if _skip:
        print(f'  [{CATEGORY}] Stage 2 skipped — min class count'
              f" {_class_counts.min() if len(_class_counts) else 0} < 20")
        continue

    Xs_tr = sev_tr[FEATURE_COLS].values; ys_tr = sev_tr['failure_level'].values.astype(int) - 1
    Xs_te = sev_te[FEATURE_COLS].values; ys_te = sev_te['failure_level'].values.astype(int) - 1
    with mlflow.start_run(run_name=f'stage2_severity_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'xgboost_multiclass',
                         'stage': '2', 'num_classes': '3'})
        sev_model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                   objective='multi:softprob', num_class=3,
                                   tree_method='hist', random_state=42, n_jobs=-1)
        sev_model.fit(Xs_tr, ys_tr, verbose=False)
        sev_auc = roc_auc_score(ys_te, sev_model.predict_proba(Xs_te),
                                multi_class='ovr', average='macro')
        sev_f1 = f1_score(ys_te, sev_model.predict(Xs_te), average='macro')
        mlflow.log_metrics({'test_severity_auc': round(sev_auc, 4),
                            'test_severity_f1': round(sev_f1, 4)})
        CATEGORY_RESULTS[CATEGORY]['severity_model'] = sev_model
    print(f'  [{CATEGORY}] Stage2  AUC={sev_auc:.4f}  F1={sev_f1:.4f}')


In [ ]:
# ── CELL 31 : Ranking Metrics + Monthly Drift Check (per category) ──────────────
# Replaces recall-only threshold selection.
# Tune threshold on VAL set → evaluate results on TEST set.
# Budget guard: if F2-optimal fleet coverage > 40%, fall back to top-25% budget threshold.
# Dedup fix: always show original F2 threshold in comparison table even if overridden.

import matplotlib.pyplot as plt

DISPATCH_BUDGET_PCT   = DISPATCH_BUDGET_PCT   if "DISPATCH_BUDGET_PCT"   in dir() else 0.25
QUALITY_RECALL_FLOOR  = QUALITY_RECALL_FLOOR  if "QUALITY_RECALL_FLOOR"  in dir() else {"TVM": 0.80, "GATE": 0.70}

for CATEGORY in ["VALIDATOR"]:
    cat         = CATEGORIES[CATEGORY]
    X_val       = cat["X_val"];   y_val  = cat["y_val"]
    X_test      = cat["X_test"];  y_test = cat["y_test"]
    RECALL_FLOOR = QUALITY_RECALL_FLOOR.get(CATEGORY, 0.80)

    proba_calib = CATEGORY_RESULTS[CATEGORY].get("proba_calib")
    if proba_calib is None:
        calib_m = CATEGORY_RESULTS[CATEGORY].get("calib_model")
        proba_calib = calib_m.predict_proba(X_test)[:, 1] if calib_m else None
    if proba_calib is None:
        print(f"[{CATEGORY}] WARNING: calibrated probabilities not found — using raw LGB")
        lgb_m = CATEGORY_RESULTS[CATEGORY]["lgb_tuned_model"]
        proba_calib = lgb_m.predict_proba(X_test)[:, 1]

    # ── Val-set proba for threshold tuning ───────────────────────────────────
    calib_m     = CATEGORY_RESULTS[CATEGORY].get("calib_model")
    lgb_m       = CATEGORY_RESULTS[CATEGORY]["lgb_tuned_model"]
    val_proba   = calib_m.predict_proba(X_val)[:, 1] if calib_m else lgb_m.predict_proba(X_val)[:, 1]
    fleet_size  = len(X_test)

    # ── F2-optimal threshold on VAL set ──────────────────────────────────────
    _best_f2 = -1; _opt_thresh = 0.5
    for _t in np.linspace(0.01, 0.99, 400):
        _p   = (val_proba >= _t).astype(int)
        _pr  = precision_score(y_val, _p, zero_division=0)
        _rc  = recall_score(y_val, _p, zero_division=0)
        _f2  = (5 * _pr * _rc) / (4 * _pr + _rc + 1e-9)
        if _f2 > _best_f2:
            _best_f2 = _f2; _opt_thresh = _t

    _f2_orig_thresh = _opt_thresh          # save BEFORE any auto-guard override
    _fl_f2 = (proba_calib >= _opt_thresh).mean()

    # ── Budget threshold: top 25% of fleet ───────────────────────────────────
    _budget_thresh = float(np.quantile(proba_calib, 1 - DISPATCH_BUDGET_PCT))

    # ── Auto-guard: if F2-optimal coverage > 40%, use budget threshold ───────
    if _fl_f2 > 0.40:
        print(f"[{CATEGORY}] AUTO-GUARD: F2-optimal flags {_fl_f2:.0%} of fleet "
              f"(>40%) — overriding to budget threshold {_budget_thresh:.4f}")
        _opt_thresh = _budget_thresh

    # ── Evaluate BOTH thresholds on TEST set ─────────────────────────────────
    def _eval_thresh(label, t):
        preds  = (proba_calib >= t).astype(int)
        fleet_pct = preds.mean()
        ap  = average_precision_score(y_test, proba_calib)
        pr  = precision_score(y_test, preds, zero_division=0)
        rc  = recall_score(y_test, preds, zero_division=0)
        f2  = (5 * pr * rc) / (4 * pr + rc + 1e-9)
        f1  = 2 * pr * rc / (pr + rc + 1e-9)
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        return {"label": label, "thresh": float(t), "fleet_pct": float(fleet_pct),
                "precision": float(pr), "recall": float(rc),
                "f2": float(f2), "f1": float(f1), "ap": float(ap),
                "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

    _bgt_tag  = " [OPERATIONAL]" if abs(_opt_thresh - _budget_thresh) < 1e-8 else ""
    _m_f2     = _eval_thresh(
        f"{CATEGORY} @ {_f2_orig_thresh:.4f} (F2-opt, fleet={_fl_f2:.0%})", _f2_orig_thresh)
    _m_bgt    = _eval_thresh(
        f"{CATEGORY} @ {_budget_thresh:.4f} (budget {DISPATCH_BUDGET_PCT:.0%}){_bgt_tag}", _budget_thresh)
    _m_active = _eval_thresh(f"{CATEGORY} ACTIVE threshold", _opt_thresh)

    # ── Quality gate check ───────────────────────────────────────────────────
    _qg_pass  = _m_active["recall"] >= RECALL_FLOOR
    CATEGORY_RESULTS[CATEGORY]["qg_pass"] = _qg_pass
    CATEGORY_RESULTS[CATEGORY]["qg_recall"] = _m_active["recall"]
    CATEGORY_RESULTS[CATEGORY]["qg_floor"]  = RECALL_FLOOR

    # ── Store ────────────────────────────────────────────────────────────────
    CATEGORY_RESULTS[CATEGORY]["opt_thresh"]    = _opt_thresh
    CATEGORY_RESULTS[CATEGORY]["f2_thresh"]     = _f2_orig_thresh
    CATEGORY_RESULTS[CATEGORY]["budget_thresh"] = _budget_thresh
    CATEGORY_RESULTS[CATEGORY]["active_metrics"]= _m_active

    # ── Print comparison table ───────────────────────────────────────────────
    print(f"\n[{CATEGORY}] Threshold comparison (tuned on VAL, evaluated on TEST)")
    print(f"{'Threshold':<52} {'Fleet':>6} {'Precision':>10} {'Recall':>8} {'F2':>7} {'AP':>7}")
    print("-" * 95)
    for _m in [_m_f2, _m_bgt]:
        print(f"  {_m['label']:<50} {_m['fleet_pct']:>6.1%} {_m['precision']:>10.4f} "
              f"{_m['recall']:>8.4f} {_m['f2']:>7.4f} {_m['ap']:>7.4f}")
    print("-" * 95)

    if _qg_pass:
        print(f"  Quality gate: PASS  recall={_m_active['recall']:.4f} >= floor={RECALL_FLOOR}")
    else:
        print(f"  Quality gate: FAIL  recall={_m_active['recall']:.4f} < floor={RECALL_FLOOR}  "
              f"(model still logged — not promoted)")

    # ── MLflow: log threshold artifact ───────────────────────────────────────
    _thresh_artifact = {
        "category": CATEGORY, "active_threshold": float(_opt_thresh),
        "f2_threshold": float(_f2_orig_thresh), "budget_threshold": float(_budget_thresh),
        "dispatch_budget_pct": DISPATCH_BUDGET_PCT, "recall_floor": RECALL_FLOOR,
        "quality_gate_pass": _qg_pass,
        "active_metrics": {k: int(v) if isinstance(v, np.integer)
                               else float(v) if isinstance(v, (float, np.floating))
                               else bool(v) if isinstance(v, (bool, np.bool_))
                               else v
                           for k, v in _m_active.items()},
    }
    import json as _json_mod
    _ta_path = f"/tmp/threshold_{CATEGORY.lower()}.json"
    open(_ta_path, "w").write(_json_mod.dumps(_thresh_artifact, indent=2))

    stack_run = CATEGORY_RESULTS[CATEGORY].get("stack_run")
    if stack_run:
        # Use MlflowClient directly — avoids experiment-ID mismatch when SageMaker
        # pins MLFLOW_EXPERIMENT_ID env var to a single experiment across the loop.
        from mlflow.tracking import MlflowClient as _MLC
        _mlc = _MLC()
        _mlc.log_artifact(stack_run, _ta_path, "threshold")
        _tag = CATEGORY.lower()
        for _mk, _mv in {
            f"thresh_{_tag}_active"   : float(_opt_thresh),
            f"thresh_{_tag}_f2"       : float(_f2_orig_thresh),
            f"thresh_{_tag}_budget"   : float(_budget_thresh),
            f"recall_{_tag}_active"   : _m_active["recall"],
            f"precision_{_tag}_active": _m_active["precision"],
            f"f2_{_tag}_active"       : _m_active["f2"],
        }.items():
            _mlc.log_metric(stack_run, _mk, _mv)
        print(f"  MLflow: threshold artifact logged to run {stack_run}")

In [ ]:
# ── CELL 32 : MLflow Model Registration (per category) ──────────────────────────
# Ranking metrics (NDCG, MAP, coverage lift) complement binary classification metrics.
# Monthly drift: compute AUC/AP on TEST subset grouped by month.
# Guard: skip months with < 5 positives (too few for reliable AUC estimate).

for CATEGORY in ["VALIDATOR"]:
    cat       = CATEGORIES[CATEGORY]
    X_test    = cat["X_test"]; y_test = cat["y_test"]
    df_test   = cat.get("df_test")

    proba_calib = CATEGORY_RESULTS[CATEGORY].get("proba_calib")
    calib_m     = CATEGORY_RESULTS[CATEGORY].get("calib_model")
    lgb_m       = CATEGORY_RESULTS[CATEGORY]["lgb_tuned_model"]
    if proba_calib is None:
        proba_calib = calib_m.predict_proba(X_test)[:, 1] if calib_m else lgb_m.predict_proba(X_test)[:, 1]

    opt_thresh = CATEGORY_RESULTS[CATEGORY].get("opt_thresh", 0.5)
    preds_bin  = (proba_calib >= opt_thresh).astype(int)

    # ── Ranking metrics ───────────────────────────────────────────────────────
    _n = len(y_test)
    _sorted_idx = np.argsort(proba_calib)[::-1]
    _k = max(1, int(_n * 0.10))          # top-10% coverage
    _hits_k = y_test.iloc[_sorted_idx[:_k]].sum() if hasattr(y_test, "iloc") else y_test[_sorted_idx[:_k]].sum()
    _total_pos = y_test.sum()
    _prec_k   = float(_hits_k) / _k
    _rec_k    = float(_hits_k) / max(float(_total_pos), 1)
    _lift_k   = _prec_k / max(float(y_test.mean()), 1e-9)

    # NDCG@k
    try:
        from sklearn.metrics import ndcg_score as _ndcg
        _y_2d = y_test.values.reshape(1,-1) if hasattr(y_test, "values") else y_test.reshape(1,-1)
        _p_2d = proba_calib.reshape(1,-1)
        _ndcg_k = float(_ndcg(_y_2d, _p_2d, k=_k))
    except Exception: _ndcg_k = float("nan")

    # MAP
    try:
        _ap_score = float(average_precision_score(y_test, proba_calib))
    except Exception: _ap_score = float("nan")

    print(f"\n[{CATEGORY}] Ranking metrics (top {_k:,} of {_n:,}, {_k/_n:.0%} coverage)")
    print(f"  Precision@{_k:,} = {_prec_k:.4f}   Recall@{_k:,} = {_rec_k:.4f}")
    print(f"  Lift@{_k:,}      = {_lift_k:.2f}x    MAP = {_ap_score:.4f}   NDCG@k = {_ndcg_k:.4f}")

    CATEGORY_RESULTS[CATEGORY].update({
        "prec_k": _prec_k, "rec_k": _rec_k, "lift_k": _lift_k,
        "ndcg_k": _ndcg_k, "map":   _ap_score,
    })

    # ── Monthly drift check ────────────────────────────────────────────────────
    if df_test is not None and "date" in df_test.columns:
        _date_col = "date"
    elif df_test is not None and DATE_COL in df_test.columns:
        _date_col = DATE_COL
    else:
        _date_col = None

    if _date_col:
        _df_d = df_test.copy()
        _df_d["_month"]  = pd.to_datetime(_df_d[_date_col]).dt.to_period("M")
        _df_d["_proba"]  = proba_calib
        _df_d["_actual"] = y_test.values if hasattr(y_test, "values") else y_test

        _drift_rows = []
        for _mo, _grp in _df_d.groupby("_month"):
            _n_pos = int(_grp["_actual"].sum())
            if _n_pos < 5: continue          # skip low-signal months
            try:
                _auc = roc_auc_score(_grp["_actual"], _grp["_proba"])
                _ap  = average_precision_score(_grp["_actual"], _grp["_proba"])
                _drift_rows.append({"month": str(_mo), "n": len(_grp),
                                    "pos": _n_pos, "auc": _auc, "ap": _ap})
            except Exception: pass

        if _drift_rows:
            print(f"\n[{CATEGORY}] Monthly drift — AUC/AP on test set (min 5 positives)")
            print(f"  {'Month':<12} {'N':>7} {'Pos':>5} {'AUC':>8} {'AP':>8}")
            print("  " + "-" * 46)
            _aucs = [r["auc"] for r in _drift_rows]
            for r in _drift_rows:
                _flag = "  WARN" if r["auc"] < np.mean(_aucs) - 0.05 else ""
                print(f"  {r['month']:<12} {r['n']:>7,} {r['pos']:>5} "
                      f"{r['auc']:>8.4f} {r['ap']:>8.4f}{_flag}")
            _min_auc = min(_aucs); _max_auc = max(_aucs)
            print(f"\n  AUC range: {_min_auc:.4f} – {_max_auc:.4f}  "
                  f"drift={_max_auc-_min_auc:.4f}  "
                  + ("STABLE" if _max_auc - _min_auc < 0.05 else "WARN: drift > 0.05"))
            CATEGORY_RESULTS[CATEGORY]["drift_monthly"] = _drift_rows
        else:
            print(f"\n[{CATEGORY}] Monthly drift: no qualifying months (< 5 positives each)")
    else:
        print(f"\n[{CATEGORY}] Monthly drift: date column not available in df_test — skipping")

In [ ]:
# ── CELL 33 : Register champion in SageMaker Model Registry (per category) ──────
# Selects champion run by test_ap across all runs in the experiment.
# Registers calibrated LGB model (CalibratedClassifierCV) to MLflow registry.
# Also registers to SageMaker model registry.
# Quality gate: only promote to Staging if recall >= QUALITY_RECALL_FLOOR.
# Version tracking: archive existing Staging version before promoting new one.

import boto3, json as _json_mod
from mlflow.tracking import MlflowClient

_mlflow_client = MlflowClient()

for CATEGORY in ["VALIDATOR"]:
    EXPERIMENT_NAME     = CATEGORIES[CATEGORY]["experiment_name"]
    MLFLOW_MODEL_NAME   = CATEGORIES[CATEGORY]["model_name"]
    MODEL_REGISTRY_NAME = CATEGORIES[CATEGORY]["registry_name"]

    opt_thresh    = CATEGORY_RESULTS[CATEGORY].get("opt_thresh", 0.5)
    f2_thresh     = CATEGORY_RESULTS[CATEGORY].get("f2_thresh", opt_thresh)
    budget_thresh = CATEGORY_RESULTS[CATEGORY].get("budget_thresh", opt_thresh)
    qg_pass       = CATEGORY_RESULTS[CATEGORY].get("qg_pass", False)
    qg_recall     = CATEGORY_RESULTS[CATEGORY].get("qg_recall", 0.0)
    qg_floor      = CATEGORY_RESULTS[CATEGORY].get("qg_floor", 0.80)

    print(f"\n{'='*60}")
    print(f"[{CATEGORY}] MLflow Registration")
    print(f"  Quality gate: {'PASS' if qg_pass else 'FAIL'}"
          f"  recall={qg_recall:.4f}  floor={qg_floor}")
    print(f"{'='*60}")

    # ── 1. Find champion run by test_ap ──────────────────────────────────────
    _exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if _exp is None:
        print(f"  [WARN] Experiment '{EXPERIMENT_NAME}' not found — skipping")
        continue

    _runs = _mlflow_client.search_runs(
        experiment_ids=[_exp.experiment_id],
        filter_string="",
        order_by=["metrics.test_ap DESC"],
        max_results=50,
    )
    _champ_run = None
    for _r in _runs:
        if "test_ap" in _r.data.metrics:
            _champ_run = _r; break

    if _champ_run is None:
        print(f"  [WARN] No run with test_ap found in experiment — skipping registration")
        continue

    _champ_ap  = _champ_run.data.metrics["test_ap"]
    _champ_auc = _champ_run.data.metrics.get("test_auc", float("nan"))
    print(f"  Champion run: {_champ_run.info.run_id[:8]}...  "
          f"test_ap={_champ_ap:.4f}  test_auc={_champ_auc:.4f}")

    # ── 2. Check if calibrated model artifact exists in champion run ──────────
    _artifacts = [a.path for a in _mlflow_client.list_artifacts(_champ_run.info.run_id)]
    _calib_path = "calibrated_lgb" if "calibrated_lgb" in _artifacts else None
    if _calib_path is None:
        # Fall back to log a fresh calibrated model artifact
        _calib_m = CATEGORY_RESULTS[CATEGORY].get("calib_model")
        if _calib_m is not None:
            print(f"  Logging calibrated model to champion run {_champ_run.info.run_id[:8]}...")
            from mlflow.tracking import MlflowClient as _MLC2
            from mlflow.models import infer_signature as _isig
            _mlc2 = _MLC2(); _X_val = CATEGORIES[CATEGORY]["X_val"]
            import tempfile as _tmp2, os as _osa
            _tmp2_dir = _tmp2.mkdtemp()
            try:
                import mlflow.sklearn as _msk
                _msk.save_model(_calib_m, _osa.path.join(_tmp2_dir, "calibrated_lgb"),
                                signature=_isig(_X_val, _calib_m.predict_proba(_X_val)))
                _mlc2.log_artifacts(_champ_run.info.run_id,
                                    _osa.path.join(_tmp2_dir, "calibrated_lgb"),
                                    artifact_path="calibrated_lgb")
            except Exception as _le2: print(f"  [Warn] log calibrated model: {_le2}")
            _calib_path = "calibrated_lgb"
        else:
            print(f"  [WARN] No calibrated model available — falling back to raw LGB")

    _model_uri = f"runs:/{_champ_run.info.run_id}/{_calib_path or 'model'}"
    print(f"  Model URI  : {_model_uri}")

    # ── 3. Register to MLflow registry ───────────────────────────────────────
    try:
        _mv = mlflow.register_model(
            model_uri=_model_uri,
            name=MLFLOW_MODEL_NAME,
        )
        _new_version = _mv.version
        print(f"  Registered : {MLFLOW_MODEL_NAME} v{_new_version}")

        # Tag the new version
        _tag_dict = {
            "champion_run"    : _champ_run.info.run_id,
            "test_ap"         : f"{_champ_ap:.4f}",
            "test_auc"        : f"{_champ_auc:.4f}",
            "active_threshold": f"{opt_thresh:.4f}",
            "f2_threshold"    : f"{f2_thresh:.4f}",
            "budget_threshold": f"{budget_thresh:.4f}",
            "quality_gate"    : "PASS" if qg_pass else "FAIL",
            "recall"          : f"{qg_recall:.4f}",
            "calibration"     : "isotonic_prefit_val",
        }
        for _tk, _tv in _tag_dict.items():
            _mlflow_client.set_model_version_tag(MLFLOW_MODEL_NAME, _new_version, _tk, _tv)

        # Archive existing Staging versions before promoting
        _existing = _mlflow_client.search_model_versions(f"name='{MLFLOW_MODEL_NAME}'")
        for _ev in _existing:
            if _ev.current_stage == "Staging" and _ev.version != _new_version:
                _mlflow_client.transition_model_version_stage(
                    MLFLOW_MODEL_NAME, _ev.version, "Archived")
                print(f"  Archived   : {MLFLOW_MODEL_NAME} v{_ev.version} (was Staging)")

        # Promote to Staging only if quality gate passes
        if qg_pass:
            _mlflow_client.transition_model_version_stage(
                MLFLOW_MODEL_NAME, _new_version, "Staging")
            print(f"  Promoted   : {MLFLOW_MODEL_NAME} v{_new_version} -> Staging")
        else:
            print(f"  NOT promoted (quality gate FAIL: recall={qg_recall:.4f} < {qg_floor})")
            print(f"  Model registered as None stage — manual promotion required after fix")

    except Exception as _e:
        print(f"  [WARN] MLflow registration failed: {_e}")
        _new_version = None

    # ── 4. Log threshold JSON artifact to MLflow ──────────────────────────────
    _thresh_json = {
        "category"          : CATEGORY,
        "active_threshold"  : float(opt_thresh),
        "f2_threshold"      : float(f2_thresh),
        "budget_threshold"  : float(budget_thresh),
        "dispatch_budget_pct": DISPATCH_BUDGET_PCT,
        "recall_floor"      : qg_floor,
        "quality_gate_pass" : qg_pass,
        "test_ap"           : float(_champ_ap),
        "test_auc"          : float(_champ_auc),
        "mlflow_model_name" : MLFLOW_MODEL_NAME,
        "mlflow_version"    : int(_new_version) if _new_version else None,
    }
    _tj_path = f"/tmp/threshold_final_{CATEGORY.lower()}.json"
    open(_tj_path, "w").write(_json_mod.dumps(_thresh_json, indent=2))
    try:
        from mlflow.tracking import MlflowClient as _MLC3
        _MLC3().log_artifact(_champ_run.info.run_id, _tj_path, "deployment")
    except Exception as _e:
        print(f"  [Warn] threshold artifact log failed: {_e}")

    # ── 5. Register to SageMaker Model Registry ───────────────────────────────
    try:
        _sm_client = boto3.client("sagemaker", region_name=AWS_REGION)
        _sm_resp   = _sm_client.list_model_package_groups(
            NameContains=MODEL_REGISTRY_NAME, MaxResults=1)

        if not _sm_resp.get("ModelPackageGroupSummaryList"):
            _sm_client.create_model_package_group(
                ModelPackageGroupName=MODEL_REGISTRY_NAME,
                ModelPackageGroupDescription=f"CUBIC MARS {CATEGORY} 3d failure prediction")
            print(f"  SageMaker: Created group '{MODEL_REGISTRY_NAME}'")

        # Find existing Production version to compare
        _existing_pkgs = _sm_client.list_model_packages(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelApprovalStatus="Approved",
            SortBy="CreationTime", SortOrder="Descending",
            MaxResults=1).get("ModelPackageSummaryList", [])

        _approve = "Approved" if qg_pass else "PendingManualApproval"
        _mp_input = {
            "ModelPackageGroupName": MODEL_REGISTRY_NAME,
            "ModelPackageDescription": (
                f"test_ap={_champ_ap:.4f} test_auc={_champ_auc:.4f} "
                f"recall={qg_recall:.4f} thresh={opt_thresh:.4f} "
                f"calib=isotonic QG={'PASS' if qg_pass else 'FAIL'}"
            ),
            "ModelApprovalStatus": _approve,
            "CustomerMetadataProperties": {
                k: str(v) for k, v in _thresh_json.items()
                if v is not None and isinstance(v, (str, int, float, bool))
            },
        }
        _mp_resp = _sm_client.create_model_package(**_mp_input)
        _mp_arn  = _mp_resp["ModelPackageArn"]
        print(f"  SageMaker: Registered {_approve}")
        print(f"    ARN: {_mp_arn}")

        CATEGORY_RESULTS[CATEGORY]["sm_model_package_arn"] = _mp_arn

    except Exception as _e:
        print(f"  [Warn] SageMaker registration skipped: {_e}")
        CATEGORY_RESULTS[CATEGORY]["sm_model_package_arn"] = None

    CATEGORY_RESULTS[CATEGORY]["mlflow_version"]  = _new_version
    CATEGORY_RESULTS[CATEGORY]["champ_ap"]        = _champ_ap
    # Keys expected by cell-23-smreg
    CATEGORY_RESULTS[CATEGORY]["champion_run_id"] = _champ_run.info.run_id
    CATEGORY_RESULTS[CATEGORY]["champion_name"]   = MLFLOW_MODEL_NAME
    CATEGORY_RESULTS[CATEGORY]["champion_ap"]     = _champ_ap
    CATEGORY_RESULTS[CATEGORY]["champion_auc"]    = _champ_auc

print("\nRegistration complete for all categories.")

In [ ]:
# ── CELL 34 : Approve + deploy SageMaker Endpoints (per category) ───────────────
for CATEGORY in ["VALIDATOR"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    import tarfile, sagemaker.image_uris as sm_images
    CHAMPION_RUN_ID  = CATEGORY_RESULTS[CATEGORY]['champion_run_id']
    CHAMPION_NAME    = CATEGORY_RESULTS[CATEGORY]['champion_name']
    CHAMPION_AP      = CATEGORY_RESULTS[CATEGORY]['champion_ap']
    CHAMPION_AUC     = CATEGORY_RESULTS[CATEGORY]['champion_auc']
    MLFLOW_VERSION   = CATEGORY_RESULTS[CATEGORY]['mlflow_version']

    # MLflow >= 2.17 uses Model IDs (m-xxx path), so runs:/<id>/artifact can resolve to
    # an empty local path. Use the in-memory model from CATEGORY_RESULTS when available.
    _in_mem = CATEGORY_RESULTS[CATEGORY].get("lgb_tuned_model")
    if _in_mem is not None:
        champion_model = _in_mem
        print(f"  [{CATEGORY}] Using in-memory lgb_tuned_model (skipping MLflow download)")
    else:
        try:
            champion_model = mlflow.sklearn.load_model(f'runs:/{CHAMPION_RUN_ID}/calibrated_lgb')
        except Exception as _lm_e:
            print(f"  [WARN] mlflow.sklearn.load_model failed: {_lm_e}")
            try:
                import mlflow.lightgbm as _mlgb
                champion_model = _mlgb.load_model(f'runs:/{CHAMPION_RUN_ID}/calibrated_lgb')
            except Exception as _lm_e2:
                raise RuntimeError(
                    f"Cannot load champion model: sklearn err={_lm_e}  lgb err={_lm_e2}"
                ) from _lm_e2
    joblib.dump(champion_model, f'/tmp/ps1_{tag}_champion.joblib')
    joblib.dump(FEATURE_COLS,   f'/tmp/ps1_{tag}_feature_cols.joblib')
    test_metrics = evaluate_all(champion_model, f'champion_{tag}_final')
    S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,
                          Key=f'sagemaker/ps1-3d/{tag}/metrics/test_metrics.json',
                          Body=json.dumps(test_metrics), ContentType='application/json')
    # Write requirements.txt so the sklearn container installs LightGBM on startup.
    # Write inference.py so the container loads the non-standard filename correctly.
    _req_path = f'/tmp/requirements_{tag}.txt'
    _inf_path = f'/tmp/inference_{tag}.py'
    open(_req_path, 'w').write("lightgbm>=4.0\nscikit-learn>=1.0\n")
    open(_inf_path, 'w').write(f'''import os, joblib
import numpy as np
import pandas as pd

def model_fn(model_dir):
    """Load champion model + feature cols from the model directory."""
    model   = joblib.load(os.path.join(model_dir, "ps1_{tag}_champion.joblib"))
    f_cols  = joblib.load(os.path.join(model_dir, "ps1_{tag}_feature_cols.joblib"))
    return {{"model": model, "feature_cols": f_cols}}

def predict_fn(input_data, model_bundle):
    model, cols = model_bundle["model"], model_bundle["feature_cols"]
    if hasattr(input_data, "values"):
        X = input_data[cols].values
    else:
        X = input_data
    return model.predict_proba(X)[:, 1].tolist()

def input_fn(request_body, content_type):
    if content_type == "application/json":
        import json
        data = json.loads(request_body)
        return pd.DataFrame(data)
    elif content_type == "text/csv":
        from io import StringIO
        return pd.read_csv(StringIO(request_body), header=None)
    raise ValueError(f"Unsupported content type: {{content_type}}")

def output_fn(prediction, accept):
    import json
    return json.dumps(prediction), "application/json"
''')

    with tarfile.open(f'/tmp/model_{tag}.tar.gz', 'w:gz') as tar:
        tar.add(f'/tmp/ps1_{tag}_champion.joblib',    arcname=f'ps1_{tag}_champion.joblib')
        tar.add(f'/tmp/ps1_{tag}_feature_cols.joblib', arcname=f'ps1_{tag}_feature_cols.joblib')
        tar.add(_req_path, arcname='requirements.txt')
        tar.add(_inf_path, arcname='inference.py')
    print(f'  [{CATEGORY}] tar.gz includes requirements.txt + inference.py (LightGBM fix)')
    MODEL_S3_KEY = f'sagemaker/ps1-3d/{tag}/model-v{MLFLOW_VERSION}/model.tar.gz'
    S3_CLIENT.upload_file(f'/tmp/model_{tag}.tar.gz', ARTIFACT_BUCKET, MODEL_S3_KEY)
    MODEL_DATA_URL = f's3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}'

    image_uri = sm_images.retrieve(framework='sklearn', region=REGION, version='1.2-1',
                                    instance_type='ml.m5.large', image_scope='inference')
    try:
        SM_CLIENT.create_model_package_group(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageGroupDescription=f'PS1-3d {CATEGORY} — Chicago Ventra. will_fail_3d. v6.')
    except Exception: pass
    resp = SM_CLIENT.create_model_package(
        ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageDescription=f'{CHAMPION_NAME} | AP={CHAMPION_AP:.4f} | MLflow v{MLFLOW_VERSION}',
        InferenceSpecification={
            'Containers': [{'Image': image_uri, 'ModelDataUrl': MODEL_DATA_URL}],
            'SupportedContentTypes': ['text/csv', 'application/json'],
            'SupportedResponseMIMETypes': ['application/json'],
            'SupportedTransformInstanceTypes': ['ml.m5.large'],
            'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large'],
        },
        ModelApprovalStatus='PendingManualApproval',
        ModelMetrics={'ModelQuality': {'Statistics': {'ContentType': 'application/json',
            'S3Uri': f's3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/metrics/test_metrics.json'}}},
        CustomerMetadataProperties={
            'ps': 'PS1_3d', 'category': CATEGORY, 'target': 'will_fail_3d', 'city': 'chicago',
            'algorithm': CHAMPION_NAME, 'test_ap': str(round(CHAMPION_AP, 4)),
            'test_auc': str(round(CHAMPION_AUC, 4)),
            'mlflow_version': str(MLFLOW_VERSION), 'mlflow_run_id': CHAMPION_RUN_ID,
            'n_features': str(len(FEATURE_COLS))})
    SM_MODEL_PKG_ARN = resp['ModelPackageArn']
    CATEGORY_RESULTS[CATEGORY].update({
        'sm_pkg_arn': SM_MODEL_PKG_ARN, 'model_data_url': MODEL_DATA_URL,
        'champion_model': champion_model, 'X_test': cat['X_test'], 'y_test': cat['y_test'],
        'X_train': cat['X_train'],
    })
    print(f'  [{CATEGORY}] SM Package: ...{SM_MODEL_PKG_ARN[-40:]}')

print('\nAll categories registered in MLflow + SageMaker.')
for cn, res in CATEGORY_RESULTS.items():
    print(f"  {cn:<6}  champion={res['champion_name']:<30}  ",
          f"AUC={res['champion_auc']:.4f}  AP={res['champion_ap']:.4f}  thresh={res.get('opt_thresh', res.get('threshold', 0)):.3f}")


In [ ]:
# ── CELL 35 : Final Summary ─────────────────────────────────────────────────────
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

MQ_MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    mq = ModelQualityMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                              sagemaker_session=SM_SESSION)
    y_prob_test = res["champion_model"].predict_proba(res["X_test"])[:,1]
    mq_df = pd.DataFrame({"prediction":y_prob_test, "probability":y_prob_test,
                           "label":res["y_test"].astype(str)})
    mq_df.to_csv(f"/tmp/mq_baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/mq_baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv")
    # Fix sagemaker circular import on Python 3.12: experiments._utils needs sagemaker.Session
    import sagemaker as _sm_pkg
    if not hasattr(_sm_pkg, 'Session'):
        from sagemaker.session import Session as _SMSession
        _sm_pkg.Session = _SMSession
    mq.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/output/",
        problem_type="BinaryClassification",
        inference_attribute="prediction", probability_attribute="probability",
        ground_truth_attribute="label", probability_threshold_attribute="0.5", wait=True)
    mq.create_monitoring_schedule(
        monitor_schedule_name=f"chicago-ps1-3d-{tag}-model-quality",
        endpoint_input=cat["endpoint"],
        ground_truth_input=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/",
        problem_type="BinaryClassification",
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-output/",
        constraints=mq.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Model Quality Monitor: chicago-ps1-3d-{tag}-model-quality")
    print(f"  Ground truth: s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/")
    MQ_MONITORS[CATEGORY] = mq


In [ ]:
# ── CELL 36 : Cross-wired PS1 output at device + component/serial-ID level ────
# Expands device-level PS1 predictions to component/serial-ID grain so that
# dashboards and ServiceNow API calls can be made at BOTH device and component
# resolution.  Joins in:
#   • hw_config_current  → COMPONENT_SERIAL_NBR, COMPONENT_TYPE, COMPONENT_PART_NBR
#   • device_ps2_chains  → cascade / coordinated-failure signals
#   • device_ps5_component → component-level MTTR / reliability tier
# Output written to s3://.../gold/device_ps1_cross_wired_daily  (parquet, append-by-date)
# ServiceNow posting: one row per (DEVICE_KEY, COMPONENT_SERIAL_NBR, transit_day)
# Scope: VALIDATOR (BMV) devices only. Component join via hw_config_current.

import json

S3_OPTS = {"client_kwargs": {"region_name": "us-east-1"}}
GOLD_PREFIX_KEY   = GOLD_BASE  # chicago/gold — defined in CELL 3
SILVER_PREFIX_KEY = SILVER_PREFIX  # chicago/silver — defined in CELL 3

# ── 1. Collect latest inference predictions from CATEGORY_RESULTS ──────────────
all_device_preds = []
for cat, res in CATEGORY_RESULTS.items():
    # evaluate_all() must have stored 'predictions' DataFrame in res
    if "predictions" not in res or res["predictions"] is None:
        print(f"[{cat}] No predictions found — run evaluate_all() cells first.")
        continue
    pred_df = res["predictions"].copy()
    pred_df["device_category"] = cat
    # ensure required columns exist with safe defaults
    for col, default in [("ps1_fail_prob", 0.0), ("ps1_predicted", 0), ("threshold_used", 0.5)]:
        if col not in pred_df.columns:
            pred_df[col] = default
    all_device_preds.append(pred_df)

if not all_device_preds:
    print("No predictions available — skipping cross-wire output.")
else:
    device_preds_df = pd.concat(all_device_preds, ignore_index=True)

    # ── 2. Join component inventory ─────────────────────────────────────────────
    try:
        hw_df = pd.read_parquet(
            f"s3://{BUCKET}/{SILVER_PREFIX_KEY}/hw_config_current",
            storage_options=S3_OPTS,
            columns=["DEVICE_KEY","COMPONENT_SERIAL_NBR","COMPONENT_TYPE",
                     "COMPONENT_PART_NBR","COMPONENT_MANUFACTURER"]
        ).dropna(subset=["DEVICE_KEY"])
        comp_preds_df = device_preds_df.merge(hw_df, on="DEVICE_KEY", how="left")
        print(f"hw_config_current joined — {len(comp_preds_df):,} rows (device→component expand)")
    except Exception as e:
        print(f"hw_config_current join skipped: {e}")
        comp_preds_df = device_preds_df.copy()

    # ── 3. Join PS2 cascade / coordinated-failure signals ───────────────────────
    try:
        ps2_df = pd.read_parquet(
            f"s3://{BUCKET}/{GOLD_PREFIX_KEY}/device_ps2_chains",
            storage_options=S3_OPTS
        )
        ps2_keep = ["DEVICE_KEY","transit_day",
                    "is_coordinated_station_failure","is_major_station_event",
                    "station_devices_failed","days_healthy_before_chain",
                    "no_prior_failure_in_window"]
        ps2_sig = (ps2_df[[c for c in ps2_keep if c in ps2_df.columns]]
                   .drop_duplicates(["DEVICE_KEY","transit_day"]))
        day_col = "transit_day" if "transit_day" in comp_preds_df.columns else "event_date"
        comp_preds_df = comp_preds_df.merge(
            ps2_sig.rename(columns={"transit_day": day_col}),
            on=["DEVICE_KEY", day_col], how="left"
        )
        print(f"PS2 cascade signals joined — coordinated failures: "
              f"{comp_preds_df.get('is_coordinated_station_failure', pd.Series()).sum()}")
    except Exception as e:
        print(f"PS2 join skipped: {e}")

    # ── 4. Join PS5 component reliability signals ────────────────────────────────
    try:
        ps5_df = pd.read_parquet(
            f"s3://{BUCKET}/{GOLD_PREFIX_KEY}/device_ps5_component",
            storage_options=S3_OPTS
        )
        ps5_keep = ["DEVICE_KEY","COMPONENT_SERIAL_NBR",
                    "ps5_risk_tier","avg_rolling_mttr_30d_min",
                    "total_failure_days_s28","last_failure_date_s28",
                    "avg_rolling_mttr_90d_min","max_downtime_ever_min"]
        ps5_sig = ps5_df[[c for c in ps5_keep if c in ps5_df.columns]].drop_duplicates(
            [c for c in ["DEVICE_KEY","COMPONENT_SERIAL_NBR"] if c in ps5_df.columns]
        )
        merge_keys = [k for k in ["DEVICE_KEY","COMPONENT_SERIAL_NBR"] if k in comp_preds_df.columns and k in ps5_sig.columns]
        comp_preds_df = comp_preds_df.merge(ps5_sig, on=merge_keys, how="left")
        print(f"PS5 component reliability joined on {merge_keys}")
    except Exception as e:
        print(f"PS5 join skipped: {e}")

    # ── 5. Anomaly flag — probability vs. device's 95th-percentile history ───────
    p95 = (device_preds_df.groupby("DEVICE_KEY")["ps1_fail_prob"]
           .quantile(0.95).rename("ps1_p95_hist"))
    comp_preds_df = comp_preds_df.merge(p95, on="DEVICE_KEY", how="left")
    comp_preds_df["is_prob_anomaly"] = (
        comp_preds_df["ps1_fail_prob"] > comp_preds_df["ps1_p95_hist"]
    ).fillna(False)

    # ── 6. Risk tier ──────────────────────────────────────────────────────────────
    def _risk_tier(p):
        if pd.isna(p):   return "UNKNOWN"
        if p >= 0.70:    return "CRITICAL"
        elif p >= 0.50:  return "HIGH"
        elif p >= 0.30:  return "MEDIUM"
        else:            return "LOW"
    comp_preds_df["ps1_risk_tier"] = comp_preds_df["ps1_fail_prob"].map(_risk_tier)

    # ── 7. SHAP top-3 features (stored in CATEGORY_RESULTS by evaluate_all) ──────
    for cat, res in CATEGORY_RESULTS.items():
        shap_rows = res.get("shap_top3_by_device")   # dict: DEVICE_KEY → [(feat,val), ...]
        if not shap_rows:
            continue
        shap_df = pd.DataFrame(
            [(dk, *([f for pair in feats[:3] for f in pair] + ["","",0,0,0][len(feats)*2:]))
             for dk, feats in shap_rows.items()],
            columns=["DEVICE_KEY","shap_feat1","shap_val1",
                     "shap_feat2","shap_val2","shap_feat3","shap_val3"]
        )
        mask = comp_preds_df["device_category"] == cat
        comp_preds_df = comp_preds_df.merge(shap_df, on="DEVICE_KEY", how="left", suffixes=("","_shap"))

    # ── 8. Write ServiceNow-ready parquet to S3 ──────────────────────────────────
    day_col = "transit_day" if "transit_day" in comp_preds_df.columns else "event_date"
    today_str = pd.Timestamp.today().strftime("%Y-%m-%d")
    out_path  = f"s3://{BUCKET}/{GOLD_PREFIX_KEY}/device_ps1_cross_wired_daily"
    comp_preds_df.to_parquet(out_path, index=False, storage_options=S3_OPTS)

    # ── 9. Summary ────────────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"PS1 Cross-wired Output  →  {out_path}")
    print(f"{'='*70}")
    print(f"Rows (device × component × day) : {len(comp_preds_df):,}")
    unique_devices = comp_preds_df["DEVICE_KEY"].nunique()
    unique_comps   = comp_preds_df["COMPONENT_SERIAL_NBR"].nunique() if "COMPONENT_SERIAL_NBR" in comp_preds_df.columns else "N/A"
    print(f"Unique devices                  : {unique_devices:,}")
    print(f"Unique component serial IDs     : {unique_comps:,}" if isinstance(unique_comps,int) else f"Unique component serial IDs     : {unique_comps}")
    print(f"\nRisk tier distribution:")
    print(comp_preds_df["ps1_risk_tier"].value_counts().to_string())
    print(f"\nAnomaly devices (prob > p95)    : {comp_preds_df['is_prob_anomaly'].sum():,}")
    print(f"\nColumns in output ({len(comp_preds_df.columns)}):")
    col_groups = {
        "IDs"        : ["DEVICE_KEY","COMPONENT_SERIAL_NBR","COMPONENT_TYPE","COMPONENT_PART_NBR","FACILITY_ID","device_category",day_col],
        "PS1"        : ["ps1_fail_prob","ps1_predicted","threshold_used","ps1_risk_tier","is_prob_anomaly"],
        "PS2"        : ["is_coordinated_station_failure","is_major_station_event","station_devices_failed","days_healthy_before_chain"],
        "PS5"        : ["ps5_risk_tier","avg_rolling_mttr_30d_min","total_failure_days_s28","last_failure_date_s28"],
        "SHAP"       : ["shap_feat1","shap_val1","shap_feat2","shap_val2","shap_feat3","shap_val3"],
    }
    for grp, cols in col_groups.items():
        present = [c for c in cols if c in comp_preds_df.columns]
        print(f"  {grp:<12}: {present}")
    print(f"\nServiceNow API key: DEVICE_KEY → CI sys_id lookup via cmdb_ci_hardware.list")
    print(f"                    COMPONENT_SERIAL_NBR → serial_number on cmdb_ci_hardware")

In [ ]:
# ── FINAL CELL : Save notebook as HTML + upload to S3 ─────────────
import pandas as _pd_s

print("=" * 72)
print("  CUBIC MARS PS1 — 3-Day Failure Prediction  (TVM + GATE)")
print("  Architecture: AWS v6 | SageMaker + MLflow Feature Store")
print("=" * 72)

_rows = []
for CATEGORY in ["VALIDATOR"]:
    _r = CATEGORY_RESULTS.get(CATEGORY, {})
    _am = _r.get("active_metrics", {})
    _rows.append({
        "Category"  : CATEGORY,
        "Test AUC"  : f"{_am.get('ap', float('nan')):.4f}",
        "Recall"    : f"{_am.get('recall', float('nan')):.4f}",
        "Precision" : f"{_am.get('precision', float('nan')):.4f}",
        "F2"        : f"{_am.get('f2', float('nan')):.4f}",
        "Brier"     : f"{_r.get('brier_calib', float('nan')):.4f}",
        "Threshold" : f"{_r.get('opt_thresh', float('nan')):.4f}",
        "Budget 25%": f"{_r.get('budget_thresh', float('nan')):.4f}",
        "QG"        : "PASS" if _r.get("qg_pass") else "FAIL",
        "MLflow v"  : str(_r.get("mlflow_version", "?")),
        "Ensemble"  : f"{_r.get('best_ens','LGB')} {'(used)' if _r.get('use_ensemble') else '(LGB wins)'}",
    })
_df_s = _pd_s.DataFrame(_rows)
print("\nModel Performance")
print(_df_s.to_string(index=False))

print("\n\nPer-Category Budget Thresholds")
for CATEGORY in ["VALIDATOR"]:
    _r = CATEGORY_RESULTS.get(CATEGORY, {})
    _opt = _r.get("opt_thresh", float("nan"))
    _bgt = _r.get("budget_thresh", float("nan"))
    _f2  = _r.get("f2_thresh", float("nan"))
    _op  = "auto-guard OVERRIDE" if abs(_opt - _bgt) < 1e-8 and abs(_f2 - _opt) > 1e-8 else "F2-optimal active"
    print(f"  {CATEGORY:<6}: F2={_f2:.4f}  Budget={_bgt:.4f}  Active={_opt:.4f}  [{_op}]")

print("\n\nCalibration (Brier Score — lower is better)")
for CATEGORY in ["VALIDATOR"]:
    _r = CATEGORY_RESULTS.get(CATEGORY, {})
    print(f"  {CATEGORY:<6}: raw={_r.get('brier_raw', float('nan')):.4f}"
          f"  calibrated={_r.get('brier_calib', float('nan')):.4f}"
          + ("  improved" if _r.get("brier_calib", 1) < _r.get("brier_raw", 1) else "  no change"))

print("\n\nMonthly Drift Status")
for CATEGORY in ["VALIDATOR"]:
    _r   = CATEGORY_RESULTS.get(CATEGORY, {})
    _dr  = _r.get("drift_monthly", [])
    if not _dr:
        print(f"  {CATEGORY}: no drift data"); continue
    _aucs = [m["auc"] for m in _dr]
    _rng  = max(_aucs) - min(_aucs)
    print(f"  {CATEGORY}: months={len(_dr)}  AUC range={min(_aucs):.4f}–{max(_aucs):.4f}"
          f"  drift={_rng:.4f}  {'STABLE' if _rng < 0.05 else 'WARN'}")

print("\n\nMLflow / SageMaker Registration")
for CATEGORY in ["VALIDATOR"]:
    _r   = CATEGORY_RESULTS.get(CATEGORY, {})
    _ver = _r.get("mlflow_version", "?")
    _arn = _r.get("sm_model_package_arn", "not registered")
    _qg  = "PASS" if _r.get("qg_pass") else "FAIL"
    print(f"  {CATEGORY:<6}: MLflow v{_ver}  QG={_qg}  SageMaker ARN: {_arn or 'not registered'}")

print("\n" + "=" * 72)
print("  Run complete.")
print("=" * 72)